# **CTPredict Estimation: Missing Value Data Build (v0.1)**

This notebook is the analytical build notebook for the `estimation` module. It prepares the estimation data foundation described in `docs/architecture_estimation.md`.

The first objective is to build a reproducible, auditable data contract without modifying the source files. The primary source is `data/data_clinpred.csv`; supporting sources such as `data/countries.txt` are read-only inputs.

Outputs created by later steps should go to a dedicated estimation folder, not back into the raw or primary source data.

#### <REF:ENV_CONFIG>

> #### **1. Development Environment Configuration**
> Configure notebook behavior, reproducibility seeds, warnings, and display defaults. This mirrors the existing audit/production notebooks while keeping the estimation module separate.

In [1]:
# <REF:ENV_CONFIG_CODE>
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
NOTEBOOK_VERSION = "estimation_v0.1"

print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"Random state: {RANDOM_STATE}")

Notebook version: estimation_v0.1
Random state: 42


#### <REF:PATH_RESOLUTION>

> #### **2. Project Path Resolution**
> Resolve project paths dynamically so the notebook works from `notebooks/` or the repository root. Estimation outputs are isolated under `data/processed/estimation/`.

In [2]:
# <REF:PATH_RESOLUTION_CODE>
from pathlib import Path
import sys

current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "GEMINI.md").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "GEMINI.md").exists():
    raise RuntimeError("Could not locate project root containing GEMINI.md")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_DIR = project_root / "data"
NOTEBOOKS_DIR = project_root / "notebooks"
ESTIMATION_OUTPUT_DIR = DATA_DIR / "processed" / "estimation"
ESTIMATION_MODEL_DIR = project_root / "models" / "estimation"

print(f"Project root: {project_root}")
print(f"Primary data dir: {DATA_DIR}")
print(f"Estimation output dir: {ESTIMATION_OUTPUT_DIR}")
print(f"Estimation model dir: {ESTIMATION_MODEL_DIR}")

Project root: /home/delaunan/code/delaunan/clintrialpredict
Primary data dir: /home/delaunan/code/delaunan/clintrialpredict/data
Estimation output dir: /home/delaunan/code/delaunan/clintrialpredict/data/processed/estimation
Estimation model dir: /home/delaunan/code/delaunan/clintrialpredict/models/estimation


#### <REF:LIB_INIT>

> #### **3. Library Initialization**
> Import the analytical stack for data audit, model preparation, and future operational estimators. Model training is intentionally deferred until after the field contract is validated.

In [3]:
# <REF:LIB_INIT_CODE>
import json
import math
import os
from datetime import datetime

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

print("Libraries loaded.")

Libraries loaded.


#### <REF:DATA_LOAD>

> #### **4. Source Data Load**
> Load source data read-only. `data/data_clinpred.csv` remains the primary source. Country count is reconstructed from `data/countries.txt` into a separate in-memory dataframe.

In [4]:
# <REF:DATA_LOAD_CODE>
CLINPRED_PATH = DATA_DIR / "data_clinpred.csv"
COUNTRIES_PATH = DATA_DIR / "countries.txt"

if not CLINPRED_PATH.exists():
    raise FileNotFoundError(f"Missing primary data file: {CLINPRED_PATH}")

df = pd.read_csv(CLINPRED_PATH)
print(f"Loaded data_clinpred: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

country_counts = None
if COUNTRIES_PATH.exists():
    countries = pd.read_csv(COUNTRIES_PATH, sep="|", usecols=["nct_id", "name", "removed"])
    countries["removed_flag"] = countries["removed"].astype(str).str.lower().eq("t")
    country_counts = (
        countries.loc[countries["name"].notna() & ~countries["removed_flag"]]
        .groupby("nct_id")["name"]
        .nunique()
        .rename("country_count_current_extract")
        .reset_index()
    )
    print(f"Reconstructed country counts: {country_counts.shape[0]:,} NCT IDs")
else:
    print("countries.txt not found. Country count will need a fallback strategy.")

df_base = df.merge(country_counts, on="nct_id", how="left") if country_counts is not None else df.copy()
print(f"Base estimation dataframe: {df_base.shape[0]:,} rows x {df_base.shape[1]:,} columns")

Loaded data_clinpred: 34,066 rows x 157 columns


Reconstructed country counts: 523,151 NCT IDs
Base estimation dataframe: 34,066 rows x 158 columns


#### <REF:DATA_AUDIT>

> #### **5. Estimation Data Audit**
> Audit the fields needed for cost reconstruction and future operational models. This step does not fix missingness yet; it identifies what must be modelled, assumed, or excluded.

In [5]:
# <REF:DATA_AUDIT_CODE>
KEY_COLUMNS = [
    "nct_id", "start_date", "completion_date", "primary_completion_date", "overall_status",
    "phase", "enrollment", "enrollment_type", "number_of_facilities", "country_count_current_extract",
    "primary_duration_months", "is_duration_unknown", "number_of_arms", "allocation", "masking",
    "intervention_model", "primary_purpose", "has_placebo", "has_dmc", "therapeutic_area",
    "gbd_indication_name", "gbd_cause_id_3", "is_rare_disease", "therapeutic_modality",
    "administration_complexity", "endpoint_rigor", "endpoint_structure", "sponsor_tier",
    "daly_global", "daly_high_income", "chronic_ratio_global", "market_skew_index",
]

available_key_columns = [c for c in KEY_COLUMNS if c in df_base.columns]
missing_key_columns = [c for c in KEY_COLUMNS if c not in df_base.columns]

audit_rows = []
for col in available_key_columns:
    s = df_base[col]
    audit_rows.append({
        "column": col,
        "dtype": str(s.dtype),
        "missing_n": int(s.isna().sum()),
        "missing_pct": round(float(s.isna().mean()), 4),
        "n_unique": int(s.nunique(dropna=True)),
        "sample_values": ", ".join(map(str, s.dropna().astype(str).unique()[:5])),
    })

audit_df = pd.DataFrame(audit_rows).sort_values(["missing_pct", "column"], ascending=[False, True])

print("Missing expected key columns:", missing_key_columns)
display(audit_df)

print("\nStatus distribution:")
display(df_base["overall_status"].value_counts(dropna=False).rename_axis("overall_status").reset_index(name="n"))

print("\nPhase distribution:")
display(df_base["phase"].value_counts(dropna=False).rename_axis("phase").reset_index(name="n"))

Missing expected key columns: []


,column,dtype,missing_n,missing_pct,n_unique,sample_values
9,country_count_current_extract,float64,2243,0.0658,55,"1.0, 2.0, 6.0, 18.0, 8.0"
2,completion_date,object,318,0.0093,5615,"2017-06-30, 2020-01-31, 2023-04-29, 2020-02-04, 2016-12-31"
3,primary_completion_date,object,20,0.0006,5454,"2017-04-30, 2016-10-31, 2023-04-29, 2020-02-04, 2016-12-31"
6,enrollment,float64,14,0.0004,2108,"1500.0, 73.0, 1072.0, 558.0, 55.0"
7,enrollment_type,object,14,0.0004,2,"ACTUAL, ESTIMATED"
24,administration_complexity,object,0,0.0000,3,"ROUTINE_INFUSION, SIMPLE_ORAL, INTENSIVE_MANAGEMENT"
13,allocation,object,0,0.0000,3,"RANDOMIZED, UNKNOWN, NON_RANDOMIZED"
30,chronic_ratio_global,float64,0,0.0000,274,"0.1228760815403974, 0.0257271429663846, 0.3215072115878451, 0.9999999999973176, 0.9999999999882117"
28,daly_global,float64,0,0.0000,275,"700.1606551062014, 82.92059340925763, 114.79343004016508, 37.27844240312, 8.48283978185"
29,daly_high_income,float64,0,0.0000,273,"60.61706302234244, 95.7208809157566, 34.85395562291364, 53.844427156840005, 6.96632588303"



Status distribution:


,overall_status,n
0,COMPLETED,20719
1,TERMINATED,4195
2,RECRUITING,4134
3,ACTIVE_NOT_RECRUITING,2487
4,WITHDRAWN,1245
5,NOT_YET_RECRUITING,1139
6,ENROLLING_BY_INVITATION,147



Phase distribution:


,phase,n
0,PHASE2,14387
1,PHASE3,13884
2,PHASE1/PHASE2,4739
3,PHASE2/PHASE3,1056


#### <REF:FIELD_CONTRACT>

> #### **6. Field Contract: What Can Be Used For What?**
> Classify fields before any modelling. This is the anti-leakage and simplicity contract for the estimation build.

In [6]:
# <REF:FIELD_CONTRACT_CODE>
field_contract_rows = [
    ("nct_id", "identity", "direct", "Asset identifier."),
    ("brief_title", "identity", "direct", "Display title if available."),
    ("official_title", "identity", "direct", "Detailed display title if needed."),
    ("lead_sponsor_canonical", "identity/context", "direct", "Sponsor display and optional model feature."),
    ("sponsor_tier", "context/feature", "direct_feature", "Optional operational scale feature; not a direct cost claim."),
    ("phase", "stage", "direct_feature", "Core current-stage and future-path driver."),
    ("therapeutic_area", "clinical_context", "direct_feature", "TA feature and cost/market grouping."),
    ("gbd_indication_name", "clinical_context", "direct_feature", "Indication display and grouping."),
    ("gbd_cause_id_3", "clinical_context", "direct_feature", "Stable indication grouping feature."),
    ("is_rare_disease", "clinical_context", "direct_feature", "Recruitment burden and pricing-power proxy."),
    ("start_date", "calendar", "direct_feature", "Needed for scenario eligibility and elapsed duration."),
    ("completion_date", "calendar", "target_derivation_or_planned", "Use with start_date to derive completed total trial duration target; planned/expected end date for scenario timing when not completed."),
    ("primary_completion_date", "calendar", "fallback_target_derivation_or_planned", "Fallback date for duration target or planned endpoint timing; use carefully."),
    ("primary_duration_months", "endpoint_duration", "direct_feature", "Endpoint/follow-up duration proxy; not the total trial-duration target for cost."),
    ("is_duration_unknown", "duration", "feature", "Quality flag for duration handling."),
    ("enrollment", "operational_scale", "target_or_lower_bound", "Completed ACTUAL enrollment is a final target; ongoing ACTUAL is a lower bound; ESTIMATED is planned/expected scale."),
    ("enrollment_type", "operational_scale", "quality_flag", "Distinguishes ACTUAL vs ESTIMATED enrollment."),
    ("number_of_facilities", "operational_scale", "target_or_lower_bound", "Completed trials provide the cleanest site-count target; ongoing values are lower-bound/current-extract approximations."),
    ("country_count_current_extract", "operational_scale", "target_or_lower_bound", "US-agnostic geography driver reconstructed from countries.txt; completed trials provide cleanest target, ongoing values are lower-bound/current-extract approximations."),
    ("number_of_arms", "design_complexity", "direct_feature", "Operational complexity and model feature."),
    ("allocation", "design_complexity", "direct_feature", "Randomization / design complexity."),
    ("masking", "design_complexity", "direct_feature", "Blinding complexity."),
    ("intervention_model", "design_complexity", "direct_feature", "Design complexity."),
    ("has_placebo", "design_complexity", "direct_feature", "Comparator complexity."),
    ("has_dmc", "design_complexity", "direct_feature", "Safety oversight complexity."),
    ("therapeutic_modality", "product_complexity", "direct_feature", "Product and intervention cost proxy."),
    ("administration_complexity", "product_complexity", "direct_feature", "Treatment delivery cost proxy."),
    ("endpoint_rigor", "endpoint_complexity", "direct_feature", "Endpoint cost/follow-up proxy."),
    ("endpoint_structure", "endpoint_complexity", "direct_feature", "Endpoint cost/follow-up proxy."),
    ("overall_status", "outcome/status", "do_not_show_as_future_outcome", "Use for target construction/audit; scenario treats selected assets as ongoing."),
    ("target", "completion_risk", "existing_model_target_only", "Existing completion-risk target; do not use as an operational scale feature."),
    ("daly_global", "market_potential", "future_market_feature", "Later market/pricing-power layer, separate from cost."),
    ("daly_high_income", "market_potential", "future_market_feature", "Later market/pricing-power layer, separate from cost."),
    ("chronic_ratio_global", "market_potential", "future_market_feature", "Later market/pricing-power layer, separate from cost."),
    ("market_skew_index", "market_potential", "future_market_feature", "Later market/pricing-power layer, separate from cost."),
]

field_contract = pd.DataFrame(
    field_contract_rows,
    columns=["column", "domain", "use_class", "rationale"],
)
field_contract["available"] = field_contract["column"].isin(df_base.columns)

display(field_contract)

print("Unavailable contract fields:")
display(field_contract.loc[~field_contract["available"]])

,column,domain,use_class,rationale,available
0,nct_id,identity,direct,Asset identifier.,True
1,brief_title,identity,direct,Display title if available.,True
2,official_title,identity,direct,Detailed display title if needed.,True
3,lead_sponsor_canonical,identity/context,direct,Sponsor display and optional model feature.,True
4,sponsor_tier,context/feature,direct_feature,Optional operational scale feature; not a direct cost claim.,True
5,phase,stage,direct_feature,Core current-stage and future-path driver.,True
6,therapeutic_area,clinical_context,direct_feature,TA feature and cost/market grouping.,True
7,gbd_indication_name,clinical_context,direct_feature,Indication display and grouping.,True
8,gbd_cause_id_3,clinical_context,direct_feature,Stable indication grouping feature.,True
9,is_rare_disease,clinical_context,direct_feature,Recruitment burden and pricing-power proxy.,True


Unavailable contract fields:


,column,domain,use_class,rationale,available


#### <REF:OPERATIONAL_RULES>

> #### **6.B Operational Scale Handling Rules**
> These rules clarify how planned, actual, completed, and ongoing values should be interpreted before model-ready target construction. They keep the approach country-agnostic and prevent accidental use of partial ongoing values as final truth.


In [7]:
# <REF:OPERATIONAL_RULES_CODE>
operational_scale_rules = pd.DataFrame([
    {
        "quantity": "enrollment",
        "completed_actual_rule": "Use as final target when overall_status == COMPLETED and enrollment_type == ACTUAL.",
        "ongoing_actual_rule": "Use as observed/current lower bound only; predicted final enrollment must be >= this value.",
        "estimated_rule": "Treat as planned/expected scale feature; do not assume it is final actual enrollment.",
        "mvp_note": "Keep enrollment_type in target audits and model features so ACTUAL vs ESTIMATED is explicit.",
    },
    {
        "quantity": "site_count",
        "completed_actual_rule": "Use number_of_facilities from completed trials as the cleanest final target.",
        "ongoing_actual_rule": "Use current-extract site count as lower-bound approximation; predicted final sites must be >= observed if available.",
        "estimated_rule": "No separate planned site-count field is currently present in data_clinpred.csv.",
        "mvp_note": "Validate site-count outliers before training.",
    },
    {
        "quantity": "country_count",
        "completed_actual_rule": "Use reconstructed country count from completed trials as the cleanest final target.",
        "ongoing_actual_rule": "Use current-extract country count as lower-bound approximation; predicted final countries must be >= observed if available.",
        "estimated_rule": "No separate planned country-count field is currently present in data_clinpred.csv.",
        "mvp_note": "Use country count, not includes_us, as the geography driver to keep the method US-agnostic.",
    },
    {
        "quantity": "duration",
        "completed_actual_rule": "Use date-derived total duration from start_date to completion_date, with primary_completion_date as fallback, after sanity checks.",
        "ongoing_actual_rule": "Elapsed duration at committee date is a lower bound; predicted final duration must be >= elapsed duration.",
        "estimated_rule": "Planned completion dates can inform expected duration but must be treated as uncertain.",
        "mvp_note": "primary_duration_months is an endpoint/follow-up duration proxy, not the total trial-duration target for cost.",
    },
])

display(operational_scale_rules)


,quantity,completed_actual_rule,ongoing_actual_rule,estimated_rule,mvp_note
0,enrollment,Use as final target when overall_status == COMPLETED and enrollment_type == ACTUAL.,Use as observed/current lower bound only; predicted final enrollment must be >= this value.,Treat as planned/expected scale feature; do not assume it is final actual enrollment.,Keep enrollment_type in target audits and model features so ACTUAL vs ESTIMATED is explicit.
1,site_count,Use number_of_facilities from completed trials as the cleanest final target.,Use current-extract site count as lower-bound approximation; predicted final sites must be >= observed if available.,No separate planned site-count field is currently present in data_clinpred.csv.,Validate site-count outliers before training.
2,country_count,Use reconstructed country count from completed trials as the cleanest final target.,Use current-extract country count as lower-bound approximation; predicted final countries must be >= observed if ava...,No separate planned country-count field is currently present in data_clinpred.csv.,"Use country count, not includes_us, as the geography driver to keep the method US-agnostic."
3,duration,"Use date-derived total duration from start_date to completion_date, with primary_completion_date as fallback, after ...",Elapsed duration at committee date is a lower bound; predicted final duration must be >= elapsed duration.,Planned completion dates can inform expected duration but must be treated as uncertain.,"primary_duration_months is an endpoint/follow-up duration proxy, not the total trial-duration target for cost."


#### <REF:TARGET_READINESS_FLAGS>

> #### **6.C Target-Readiness Flags**
> Create explicit boolean flags before target construction. These flags make it auditable which records can train final-scale models, which records provide lower bounds for ongoing/actionable assets, and which enrollment values are planned estimates rather than final actuals.


In [8]:
# <REF:TARGET_READINESS_FLAGS_CODE>
df_step1 = df_base.copy()

for date_col in ["start_date", "completion_date", "primary_completion_date"]:
    df_step1[f"{date_col}_parsed"] = pd.to_datetime(df_step1[date_col], errors="coerce")

df_step1["duration_end_date_for_target"] = df_step1["completion_date_parsed"].fillna(
    df_step1["primary_completion_date_parsed"]
)
df_step1["total_duration_months_observed"] = (
    (df_step1["duration_end_date_for_target"] - df_step1["start_date_parsed"]).dt.days / 30.4375
)

COMPLETED_STATUS = "COMPLETED"
ONGOING_STATUSES = {
    "RECRUITING",
    "ACTIVE_NOT_RECRUITING",
    "NOT_YET_RECRUITING",
    "ENROLLING_BY_INVITATION",
}
PARTIAL_STOP_STATUSES = {"TERMINATED", "WITHDRAWN"}

status_upper = df_step1["overall_status"].astype(str).str.upper()
enrollment_type_upper = df_step1["enrollment_type"].astype(str).str.upper()

df_step1["is_completed_status"] = status_upper.eq(COMPLETED_STATUS)
df_step1["is_ongoing_status"] = status_upper.isin(ONGOING_STATUSES)
df_step1["is_partial_stop_status"] = status_upper.isin(PARTIAL_STOP_STATUSES)

df_step1["is_completed_actual_enrollment_target"] = (
    df_step1["is_completed_status"]
    & enrollment_type_upper.eq("ACTUAL")
    & df_step1["enrollment"].notna()
    & (df_step1["enrollment"] > 0)
)
df_step1["is_ongoing_actual_enrollment_lower_bound"] = (
    df_step1["is_ongoing_status"]
    & enrollment_type_upper.eq("ACTUAL")
    & df_step1["enrollment"].notna()
    & (df_step1["enrollment"] > 0)
)
df_step1["is_estimated_planned_enrollment"] = (
    enrollment_type_upper.eq("ESTIMATED")
    & df_step1["enrollment"].notna()
    & (df_step1["enrollment"] > 0)
)

df_step1["is_completed_site_count_target"] = (
    df_step1["is_completed_status"]
    & df_step1["number_of_facilities"].notna()
    & (df_step1["number_of_facilities"] > 0)
)
df_step1["is_ongoing_site_count_lower_bound"] = (
    df_step1["is_ongoing_status"]
    & df_step1["number_of_facilities"].notna()
    & (df_step1["number_of_facilities"] > 0)
)

df_step1["is_completed_country_count_target"] = (
    df_step1["is_completed_status"]
    & df_step1["country_count_current_extract"].notna()
    & (df_step1["country_count_current_extract"] > 0)
)
df_step1["is_ongoing_country_count_lower_bound"] = (
    df_step1["is_ongoing_status"]
    & df_step1["country_count_current_extract"].notna()
    & (df_step1["country_count_current_extract"] > 0)
)

df_step1["is_completed_duration_target"] = (
    df_step1["is_completed_status"]
    & df_step1["total_duration_months_observed"].notna()
    & (df_step1["total_duration_months_observed"] > 0)
)

flag_cols = [
    "is_completed_actual_enrollment_target",
    "is_ongoing_actual_enrollment_lower_bound",
    "is_estimated_planned_enrollment",
    "is_completed_site_count_target",
    "is_ongoing_site_count_lower_bound",
    "is_completed_country_count_target",
    "is_ongoing_country_count_lower_bound",
    "is_completed_duration_target",
]

flag_summary = (
    df_step1[flag_cols]
    .sum()
    .rename("n_true")
    .reset_index()
    .rename(columns={"index": "flag"})
)
flag_summary["pct_of_rows"] = (flag_summary["n_true"] / len(df_step1)).round(4)

display(flag_summary)

print("Enrollment target/lower-bound matrix:")
display(pd.crosstab(df_step1["overall_status"], df_step1["enrollment_type"], dropna=False))


,flag,n_true,pct_of_rows
0,is_completed_actual_enrollment_target,20526,0.6025
1,is_ongoing_actual_enrollment_lower_bound,1606,0.0471
2,is_estimated_planned_enrollment,6546,0.1922
3,is_completed_site_count_target,19880,0.5836
4,is_ongoing_site_count_lower_bound,7305,0.2144
5,is_completed_country_count_target,19879,0.5835
6,is_ongoing_country_count_lower_bound,7305,0.2144
7,is_completed_duration_target,20699,0.6076


Enrollment target/lower-bound matrix:


enrollment_type,ACTUAL,ESTIMATED,NaN
overall_status,,,
ACTIVE_NOT_RECRUITING,1606,881,0
COMPLETED,20526,181,12
ENROLLING_BY_INVITATION,0,147,0
NOT_YET_RECRUITING,0,1139,0
RECRUITING,0,4134,0
TERMINATED,4142,51,2
WITHDRAWN,1232,13,0


#### <REF:STEP_1_VALIDATION>

> #### **7. Step 1 Validation Checklist**
> Before building model-ready datasets, validate this first step manually.
>
> Check:
> - `data_clinpred.csv` loads with the expected row count.
> - `country_count_current_extract` is reconstructed for most selected NCT IDs.
> - Missingness in key cost-driver columns is acceptable or clearly identified.
> - `overall_status` and `phase` distributions match expectations.
> - The field contract correctly separates direct features, operational targets/lower bounds, future market features, and fields not to expose as future outcomes.
> - Target-readiness flags identify completed final targets, ongoing lower bounds, and planned/estimated enrollment separately.
>
> Do not proceed to model training until the field contract is accepted.

#### <REF:TARGET_BUILD>

> #### **8. Operational Target Dataset Construction**
> Build separate model-ready target datasets for duration, enrollment, site count, and country count. These datasets are derived estimation inputs and must not overwrite `data/data_clinpred.csv`.
>
> The training targets use completed trials only. Ongoing trials remain available as lower-bound/application rows for later prediction.


In [9]:
# <REF:TARGET_BUILD_CODE>
TARGET_OUTPUT_DIR = ESTIMATION_OUTPUT_DIR / "targets"
TARGET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Work from the audited Step 1 dataframe with explicit target-readiness flags.
if "df_step1" not in globals():
    raise RuntimeError("Run <REF:TARGET_READINESS_FLAGS_CODE> before target construction.")

df_targets_base = df_step1.copy()

# Parse core dates for scenario-date calculations if the prior cell has not already done it.
for date_col in ["start_date", "completion_date", "primary_completion_date"]:
    parsed_col = f"{date_col}_parsed"
    if date_col in df_targets_base.columns and parsed_col not in df_targets_base.columns:
        df_targets_base[parsed_col] = pd.to_datetime(df_targets_base[date_col], errors="coerce")

if "total_duration_months_observed" not in df_targets_base.columns:
    df_targets_base["duration_end_date_for_target"] = df_targets_base["completion_date_parsed"].fillna(
        df_targets_base["primary_completion_date_parsed"]
    )
    df_targets_base["total_duration_months_observed"] = (
        (df_targets_base["duration_end_date_for_target"] - df_targets_base["start_date_parsed"]).dt.days / 30.4375
    )

# Clean numeric fields used as model targets or scale features.
df_targets_base["number_of_arms_num"] = pd.to_numeric(df_targets_base["number_of_arms"], errors="coerce")
df_targets_base["enrollment_num"] = pd.to_numeric(df_targets_base["enrollment"], errors="coerce")
df_targets_base["site_count_num"] = pd.to_numeric(df_targets_base["number_of_facilities"], errors="coerce")
df_targets_base["country_count_num"] = pd.to_numeric(df_targets_base["country_count_current_extract"], errors="coerce")
df_targets_base["endpoint_duration_months_num"] = pd.to_numeric(df_targets_base["primary_duration_months"], errors="coerce")
df_targets_base["duration_months_num"] = pd.to_numeric(df_targets_base["total_duration_months_observed"], errors="coerce")

# A conservative shared feature set for operational scale models.
# These are available or already derived without using future outcome labels as predictors.
OPERATIONAL_FEATURE_COLUMNS = [
    "nct_id",
    "phase",
    "therapeutic_area",
    "gbd_cause_id_3",
    "is_rare_disease",
    "sponsor_tier",
    "therapeutic_modality",
    "administration_complexity",
    "endpoint_rigor",
    "endpoint_structure",
    "allocation",
    "masking",
    "intervention_model",
    "primary_purpose",
    "has_placebo",
    "has_dmc",
    "number_of_arms_num",
    "enrollment_num",
    "enrollment_type",
    "site_count_num",
    "country_count_num",
    "endpoint_duration_months_num",
    "start_date_parsed",
    "is_duration_unknown",
]
OPERATIONAL_FEATURE_COLUMNS = [c for c in OPERATIONAL_FEATURE_COLUMNS if c in df_targets_base.columns]

def build_target_dataset(flag_col, target_col, target_name):
    cols = OPERATIONAL_FEATURE_COLUMNS + [flag_col, target_col]
    cols = list(dict.fromkeys([c for c in cols if c in df_targets_base.columns]))
    out = df_targets_base.loc[df_targets_base[flag_col], cols].copy()
    out = out.rename(columns={target_col: "target_value"})
    out.insert(0, "target_name", target_name)
    out["target_value_log1p"] = np.log1p(out["target_value"].astype(float))
    return out

duration_model_df = build_target_dataset(
    "is_completed_duration_target",
    "duration_months_num",
    "final_total_duration_months",
)
enrollment_model_df = build_target_dataset(
    "is_completed_actual_enrollment_target",
    "enrollment_num",
    "final_actual_enrollment",
)
site_model_df = build_target_dataset(
    "is_completed_site_count_target",
    "site_count_num",
    "final_site_count",
)
country_model_df = build_target_dataset(
    "is_completed_country_count_target",
    "country_count_num",
    "final_country_count",
)

model_target_datasets = {
    "duration": duration_model_df,
    "enrollment": enrollment_model_df,
    "site_count": site_model_df,
    "country_count": country_model_df,
}

target_dataset_summary = []
for name, data in model_target_datasets.items():
    target = data["target_value"].astype(float)
    target_dataset_summary.append({
        "dataset": name,
        "rows": len(data),
        "target_min": target.min(),
        "target_p25": target.quantile(0.25),
        "target_median": target.median(),
        "target_p75": target.quantile(0.75),
        "target_p95": target.quantile(0.95),
        "target_max": target.max(),
    })

target_dataset_summary = pd.DataFrame(target_dataset_summary)
display(target_dataset_summary)

print("Model target datasets created in memory:")
for name, data in model_target_datasets.items():
    print(f"- {name}: {data.shape[0]:,} rows x {data.shape[1]:,} columns")


,dataset,rows,target_min,target_p25,target_median,target_p75,target_p95,target_max
0,duration,20699,0.065708,11.991786,21.026694,36.566735,74.027105,190.291581
1,enrollment,20526,1.000000,59.000000,154.000000,370.000000,1097.750000,90116.000000
2,site_count,19880,1.000000,3.000000,15.000000,42.000000,140.000000,1611.000000
3,country_count,19879,1.000000,1.000000,1.000000,5.000000,18.000000,59.000000


Model target datasets created in memory:
- duration: 20,699 rows x 28 columns
- enrollment: 20,526 rows x 27 columns
- site_count: 19,880 rows x 27 columns
- country_count: 19,879 rows x 27 columns


#### <REF:TARGET_OUTLIER_AUDIT>

> #### **9. Target Outlier and Plausibility Audit**
> Review target distributions before model training. This step does not remove records yet; it identifies extreme values and proposes transparent winsorization / exclusion thresholds for validation.
>
> The goal is not to over-clean the data. The goal is to prevent a few extreme trials from dominating cost-driving operational models.


In [10]:
# <REF:TARGET_OUTLIER_AUDIT_CODE>
if "model_target_datasets" not in globals():
    raise RuntimeError("Run <REF:TARGET_BUILD_CODE> before the outlier audit.")

OUTLIER_RULES = {
    # Conservative first-pass rules. These are audit flags, not automatic exclusions yet.
    "duration": {"low": 1.0, "high": 120.0, "unit": "months"},
    "enrollment": {"low": 5.0, "high": 10000.0, "unit": "patients"},
    "site_count": {"low": 1.0, "high": 500.0, "unit": "sites"},
    "country_count": {"low": 1.0, "high": 40.0, "unit": "countries"},
}

outlier_summary_rows = []
outlier_examples = {}
for name, data in model_target_datasets.items():
    rules = OUTLIER_RULES[name]
    target = data["target_value"].astype(float)
    low_flag = target < rules["low"]
    high_flag = target > rules["high"]
    outlier_flag = low_flag | high_flag

    outlier_summary_rows.append({
        "dataset": name,
        "rows": len(data),
        "low_threshold": rules["low"],
        "high_threshold": rules["high"],
        "unit": rules["unit"],
        "n_below_low": int(low_flag.sum()),
        "n_above_high": int(high_flag.sum()),
        "n_flagged_total": int(outlier_flag.sum()),
        "pct_flagged_total": round(float(outlier_flag.mean()), 4),
    })

    example_cols = [
        c for c in [
            "nct_id", "phase", "therapeutic_area", "sponsor_tier", "enrollment_type",
            "target_value", "start_date_parsed", "completion_date_parsed", "primary_completion_date_parsed",
            "endpoint_duration_months_num", "enrollment_num", "site_count_num", "country_count_num",
        ] if c in data.columns
    ]
    outlier_examples[name] = (
        data.loc[outlier_flag, example_cols]
        .sort_values("target_value", ascending=False)
        .head(20)
        .copy()
    )

outlier_summary = pd.DataFrame(outlier_summary_rows)
display(outlier_summary)

for name, examples in outlier_examples.items():
    print(f"\nTop flagged examples for {name}:")
    display(examples)


,dataset,rows,low_threshold,high_threshold,unit,n_below_low,n_above_high,n_flagged_total,pct_flagged_total
0,duration,20699,1.0,120.0,months,67,114,181,0.0087
1,enrollment,20526,5.0,10000.0,patients,93,76,169,0.0082
2,site_count,19880,1.0,500.0,sites,0,53,53,0.0027
3,country_count,19879,1.0,40.0,countries,0,42,42,0.0021



Top flagged examples for duration:


,nct_id,phase,therapeutic_area,sponsor_tier,enrollment_type,target_value,start_date_parsed,endpoint_duration_months_num,enrollment_num,site_count_num,country_count_num
23845,NCT00777036,PHASE2,ONCOLOGY,TIER 1,ACTUAL,190.291581,2009-03-20,90.00,133.0,174,18.0
11518,NCT01560182,PHASE1/PHASE2,GENETIC,BIOTECH,ACTUAL,185.363450,2010-04-09,36.00,20.0,1,1.0
28384,NCT01065454,PHASE2,CARDIOVASCULAR,TIER 1,ACTUAL,183.293634,2010-04-14,3.68,202.0,83,18.0
16811,NCT01247207,PHASE3,MUSCULOSKELETAL,BIOTECH,ESTIMATED,182.373717,2010-11-30,96.00,270.0,34,2.0
19801,NCT01171898,PHASE1/PHASE2,ONCOLOGY,BIOTECH,ACTUAL,179.876797,2010-07-26,2.76,127.0,15,1.0
11045,NCT01294020,PHASE2,IMMUNOLOGY,TIER 1,ACTUAL,173.338809,2011-05-25,0.46,81.0,16,7.0
16877,NCT00981058,PHASE3,ONCOLOGY,TIER 1,ACTUAL,172.714579,2010-01-07,31.00,1093.0,182,26.0
25705,NCT01298323,PHASE3,ONCOLOGY,TIER 1,ACTUAL,168.542094,2011-02-25,12.00,205.0,66,20.0
3735,NCT01084863,PHASE1/PHASE2,ONCOLOGY,BIOTECH,ACTUAL,166.045175,2010-02-28,0.69,143.0,1,1.0
31661,NCT01506141,PHASE1/PHASE2,GENETIC,TIER 1,ACTUAL,164.960986,2010-08-01,165.00,15.0,9,3.0



Top flagged examples for enrollment:


,nct_id,phase,therapeutic_area,sponsor_tier,enrollment_type,target_value,start_date_parsed,endpoint_duration_months_num,site_count_num,country_count_num
7427,NCT01528449,PHASE3,INFECTIONS,BIOTECH,ACTUAL,90116.0,2011-12-31,12.00,1,1.0
6494,NCT04368728,PHASE2/PHASE3,INFECTIONS,BIOTECH,ACTUAL,46969.0,2020-04-29,6.00,175,6.0
7678,NCT05540522,PHASE3,INFECTIONS,TIER 1,ACTUAL,45789.0,2022-09-12,12.00,321,6.0
29118,NCT04526990,PHASE3,INFECTIONS,BIOTECH,ACTUAL,44247.0,2020-09-15,12.00,74,5.0
25429,NCT04510207,PHASE3,INFECTIONS,BIOTECH,ACTUAL,44101.0,2020-07-16,6.00,6,4.0
18081,NCT00861380,PHASE3,INFECTIONS,TIER 1,ACTUAL,41188.0,2009-05-04,30.00,1,1.0
9345,NCT06602024,PHASE3,INFECTIONS,BIOTECH,ACTUAL,40817.0,2024-09-16,5.95,301,11.0
24133,NCT04652102,PHASE2/PHASE3,INFECTIONS,BIOTECH,ACTUAL,39680.0,2020-12-11,12.91,46,10.0
7691,NCT04904471,PHASE3,INFECTIONS,BIOTECH,ACTUAL,39663.0,2021-06-15,6.00,24,5.0
18453,NCT05127434,PHASE2/PHASE3,INFECTIONS,BIOTECH,ACTUAL,36814.0,2021-11-17,12.00,269,23.0



Top flagged examples for site_count:


,nct_id,phase,therapeutic_area,sponsor_tier,enrollment_type,target_value,start_date_parsed,endpoint_duration_months_num,enrollment_num,country_count_num
23554,NCT01313676,PHASE3,RESPIRATORY,TIER 1,ACTUAL,1611,2011-01-25,24.00,16568.0,44.0
20288,NCT01663402,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,1388,2012-10-31,64.00,18924.0,57.0
14002,NCT02993406,PHASE3,CARDIOVASCULAR,BIOTECH,ACTUAL,1319,2016-12-22,68.00,13970.0,32.0
22644,NCT01764633,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,1284,2013-02-08,36.00,27564.0,49.0
21000,NCT01991795,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,1237,2014-02-10,40.00,19271.0,43.0
10861,NCT01126437,PHASE3,RESPIRATORY,TIER 1,ACTUAL,1191,2010-05-31,36.00,17183.0,51.0
13955,NCT01327846,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,1131,2011-04-11,72.00,10066.0,40.0
15636,NCT02929329,PHASE3,CARDIOVASCULAR,BIOTECH,ACTUAL,1033,2017-01-06,42.00,8256.0,36.0
1301,NCT05754957,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,1003,2023-04-07,42.00,14194.0,44.0
15887,NCT02164513,PHASE3,RESPIRATORY,TIER 1,ACTUAL,998,2014-06-30,11.96,10355.0,37.0



Top flagged examples for country_count:


,nct_id,phase,therapeutic_area,sponsor_tier,enrollment_type,target_value,start_date_parsed,endpoint_duration_months_num,enrollment_num,site_count_num
32705,NCT01566721,PHASE3,ONCOLOGY,TIER 1,ACTUAL,59.0,2012-05-17,36.00,2577.0,437
20288,NCT01663402,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,57.0,2012-10-31,64.00,18924.0,1388
12402,NCT01076764,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,55.0,2010-04-30,0.23,13220.0,607
10861,NCT01126437,PHASE3,RESPIRATORY,TIER 1,ACTUAL,51.0,2010-05-31,36.00,17183.0,1191
28775,NCT02296138,PHASE3,RESPIRATORY,TIER 1,ACTUAL,51.0,2015-01-13,11.86,7903.0,818
18612,NCT00968708,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,50.0,2009-09-30,41.00,5380.0,908
22644,NCT01764633,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,49.0,2013-02-08,36.00,27564.0,1284
22825,NCT03345849,PHASE3,GASTROINTESTINAL,TIER 1,ACTUAL,49.0,2017-12-07,2.76,526.0,432
7338,NCT01147250,PHASE3,CARDIOVASCULAR,TIER 1,ACTUAL,49.0,2010-06-30,25.00,6068.0,829
3578,NCT03473223,PHASE3,CARDIOVASCULAR,MID_CAP,ACTUAL,49.0,2018-03-21,2.96,18226.0,903


#### <REF:TARGET_CLEANING_POLICY>

> #### **10. Proposed Target Cleaning Policy**
> After reviewing the outlier audit, choose a simple target-cleaning policy before model benchmarking. A good MVP policy is usually:
>
> - Keep all plausible records.
> - Exclude impossible or near-impossible targets from training, but keep them in audit tables.
> - Use `log1p(target)` for skewed targets.
> - Prefer model robustness and grouped-median fallback over aggressive manual cleaning.
>
> Suggested first-pass audit thresholds are intentionally conservative and should be validated before being applied.


#### <REF:TRAINING_DATASETS>

> #### **11. Clean Training Dataset Construction**
> Apply the agreed minimal cleaning policy. Keep large global trials because they are important for portfolio cost pressure. Exclude only low implausible values from model training while retaining all records in audit outputs.
>
> Cleaning policy for MVP:
> - Exclude `duration < 1 month` from duration training.
> - Exclude `enrollment < 5 patients` from enrollment training.
> - Keep high enrollment, site, and country values; model on `log1p(target)`.
> - Keep all valid site and country target rows for now.


In [11]:
# <REF:TRAINING_DATASETS_CODE>
if "model_target_datasets" not in globals():
    raise RuntimeError("Run <REF:TARGET_BUILD_CODE> before training dataset construction.")

TRAINING_CLEANING_RULES = {
    "duration": {"min_target": 1.0, "max_target": None, "reason": "Exclude trials shorter than 1 month as implausible total-duration training targets."},
    "enrollment": {"min_target": 5.0, "max_target": None, "reason": "Exclude extremely tiny completed-enrollment targets from final-scale training."},
    "site_count": {"min_target": 1.0, "max_target": None, "reason": "Keep all positive completed site counts, including large global trials."},
    "country_count": {"min_target": 1.0, "max_target": None, "reason": "Keep all positive completed country counts, including global trials."},
}

training_datasets = {}
training_cleaning_summary = []
for name, data in model_target_datasets.items():
    rules = TRAINING_CLEANING_RULES[name]
    target = data["target_value"].astype(float)
    keep_mask = target >= rules["min_target"]
    if rules["max_target"] is not None:
        keep_mask &= target <= rules["max_target"]

    train_df = data.loc[keep_mask].copy()
    train_df["target_value_log1p"] = np.log1p(train_df["target_value"].astype(float))
    train_df["training_cleaning_rule"] = rules["reason"]

    training_datasets[name] = train_df
    training_cleaning_summary.append({
        "dataset": name,
        "source_rows": len(data),
        "training_rows": len(train_df),
        "excluded_rows": int((~keep_mask).sum()),
        "excluded_pct": round(float((~keep_mask).mean()), 4),
        "min_target_kept": train_df["target_value"].astype(float).min(),
        "median_target_kept": train_df["target_value"].astype(float).median(),
        "p95_target_kept": train_df["target_value"].astype(float).quantile(0.95),
        "max_target_kept": train_df["target_value"].astype(float).max(),
        "rule": rules["reason"],
    })

training_cleaning_summary = pd.DataFrame(training_cleaning_summary)
display(training_cleaning_summary)

for name, data in training_datasets.items():
    print(f"{name}: {data.shape[0]:,} training rows x {data.shape[1]:,} columns")


,dataset,source_rows,training_rows,excluded_rows,excluded_pct,min_target_kept,median_target_kept,p95_target_kept,max_target_kept,rule
0,duration,20699,20632,67,0.0032,1.01848,21.059548,74.130595,190.291581,Exclude trials shorter than 1 month as implausible total-duration training targets.
1,enrollment,20526,20433,93,0.0045,5.00000,154.000000,1100.000000,90116.000000,Exclude extremely tiny completed-enrollment targets from final-scale training.
2,site_count,19880,19880,0,0.0000,1.00000,15.000000,140.000000,1611.000000,"Keep all positive completed site counts, including large global trials."
3,country_count,19879,19879,0,0.0000,1.00000,1.000000,18.000000,59.000000,"Keep all positive completed country counts, including global trials."


duration: 20,632 training rows x 29 columns
enrollment: 20,433 training rows x 28 columns
site_count: 19,880 training rows x 28 columns
country_count: 19,879 training rows x 28 columns


#### <REF:MODEL_BENCHMARK_SETUP>

> #### **12. Model Benchmark Setup**
> Define a simple, reusable benchmark framework. The first benchmark should compare grouped-median baselines against gradient-boosted tabular models for each operational target.
>
> This block prepares feature typing and utility functions only. It does not yet select or save final models.


In [12]:
# <REF:MODEL_BENCHMARK_SETUP_CODE>
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    XGBRegressor = None
    print(f"XGBoost unavailable; benchmarks will use sklearn models only. Reason: {exc}")

CATEGORICAL_FEATURES = [
    "phase",
    "therapeutic_area",
    "sponsor_tier",
    "therapeutic_modality",
    "administration_complexity",
    "endpoint_rigor",
    "endpoint_structure",
    "allocation",
    "masking",
    "intervention_model",
    "primary_purpose",
    "enrollment_type",
]
NUMERIC_FEATURES = [
    "gbd_cause_id_3",
    "is_rare_disease",
    "has_placebo",
    "has_dmc",
    "number_of_arms_num",
    "enrollment_num",
    "site_count_num",
    "country_count_num",
    "endpoint_duration_months_num",
    "is_duration_unknown",
]

# Avoid target leakage by dropping target-specific observed-scale features from each model.
TARGET_FEATURE_EXCLUSIONS = {
    "duration": [],
    "enrollment": ["enrollment_num"],
    "site_count": ["site_count_num"],
    "country_count": ["country_count_num"],
}

def get_features_for_target(target_name, data):
    excluded = set(TARGET_FEATURE_EXCLUSIONS.get(target_name, []))
    cat = [c for c in CATEGORICAL_FEATURES if c in data.columns and c not in excluded]
    num = [c for c in NUMERIC_FEATURES if c in data.columns and c not in excluded]
    return cat, num

def make_preprocessor(categorical_features, numeric_features):
    return ColumnTransformer(
        transformers=[
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), categorical_features),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]), numeric_features),
        ],
        remainder="drop",
    )

def grouped_median_predict(train_df, valid_df, group_cols=("phase", "therapeutic_area")):
    global_median = train_df["target_value_log1p"].median()
    medians = train_df.groupby(list(group_cols))["target_value_log1p"].median()
    preds = []
    for _, row in valid_df.iterrows():
        key = tuple(row.get(c) for c in group_cols)
        preds.append(medians.get(key, global_median))
    return np.asarray(preds, dtype=float)

def evaluate_log_predictions(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.maximum(np.expm1(y_pred_log), 0)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {
        "mae_original": mean_absolute_error(y_true, y_pred),
        "rmse_original": rmse,
        "mae_log": mean_absolute_error(y_true_log, y_pred_log),
        "r2_log": r2_score(y_true_log, y_pred_log),
    }

print(f"XGBoost available: {HAS_XGBOOST}")
for target_name, data in training_datasets.items():
    cat, num = get_features_for_target(target_name, data)
    print(f"{target_name}: {len(cat)} categorical features, {len(num)} numeric features")


XGBoost available: True
duration: 12 categorical features, 10 numeric features
enrollment: 12 categorical features, 9 numeric features
site_count: 12 categorical features, 9 numeric features
country_count: 12 categorical features, 9 numeric features


#### <REF:MODEL_BENCHMARK_RUN>

> #### **13. Model Benchmark Run**
> Run first-pass holdout benchmarks for each operational target. These results are for review only: no fitted models or model artifacts are saved.
>
> Benchmarks compare:
> - grouped-median baseline on `phase` x `therapeutic_area`;
> - `HistGradientBoostingRegressor`;
> - `XGBRegressor` when XGBoost is available;
> - optional `RandomForestRegressor` when `RUN_RANDOM_FOREST_BENCHMARK = True`.


In [13]:
# <REF:MODEL_BENCHMARK_RUN_CODE>
if "training_datasets" not in globals():
    raise RuntimeError("Run <REF:TRAINING_DATASETS_CODE> before model benchmarking.")

BENCHMARK_RANDOM_STATE = 42
BENCHMARK_TEST_SIZE = 0.20
RUN_RANDOM_FOREST_BENCHMARK = False

MODEL_BENCHMARK_CONFIG = {
    "hist_gradient_boosting": {
        "estimator": HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.06,
            max_iter=350,
            l2_regularization=0.01,
            random_state=BENCHMARK_RANDOM_STATE,
        ),
    },
}

if HAS_XGBOOST:
    MODEL_BENCHMARK_CONFIG["xgboost"] = {
        "estimator": XGBRegressor(
            objective="reg:squarederror",
            n_estimators=450,
            learning_rate=0.04,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=BENCHMARK_RANDOM_STATE,
            n_jobs=-1,
        ),
    }

if RUN_RANDOM_FOREST_BENCHMARK:
    MODEL_BENCHMARK_CONFIG["random_forest"] = {
        "estimator": RandomForestRegressor(
            n_estimators=250,
            min_samples_leaf=5,
            max_features="sqrt",
            random_state=BENCHMARK_RANDOM_STATE,
            n_jobs=-1,
        ),
    }

benchmark_rows = []
benchmark_predictions = {}
benchmark_splits = {}

for target_name, data in training_datasets.items():
    target_data = data.copy()
    cat_features, num_features = get_features_for_target(target_name, target_data)
    feature_cols = cat_features + num_features

    if not feature_cols:
        raise RuntimeError(f"No benchmark features available for {target_name}.")

    train_df, valid_df = train_test_split(
        target_data,
        test_size=BENCHMARK_TEST_SIZE,
        random_state=BENCHMARK_RANDOM_STATE,
    )
    benchmark_splits[target_name] = {
        "train_rows": len(train_df),
        "valid_rows": len(valid_df),
        "categorical_features": cat_features,
        "numeric_features": num_features,
    }

    y_train_log = train_df["target_value_log1p"].astype(float)
    y_valid_log = valid_df["target_value_log1p"].astype(float)

    baseline_pred_log = grouped_median_predict(train_df, valid_df)
    baseline_metrics = evaluate_log_predictions(y_valid_log, baseline_pred_log)
    benchmark_rows.append({
        "target": target_name,
        "model": "grouped_median_baseline",
        "train_rows": len(train_df),
        "valid_rows": len(valid_df),
        "n_features": len(feature_cols),
        **baseline_metrics,
    })
    benchmark_predictions[(target_name, "grouped_median_baseline")] = baseline_pred_log

    for model_name, config in MODEL_BENCHMARK_CONFIG.items():
        pipeline = Pipeline([
            ("preprocessor", make_preprocessor(cat_features, num_features)),
            ("model", config["estimator"]),
        ])
        pipeline.fit(train_df[feature_cols], y_train_log)
        pred_log = pipeline.predict(valid_df[feature_cols])
        metrics = evaluate_log_predictions(y_valid_log, pred_log)
        benchmark_rows.append({
            "target": target_name,
            "model": model_name,
            "train_rows": len(train_df),
            "valid_rows": len(valid_df),
            "n_features": len(feature_cols),
            **metrics,
        })
        benchmark_predictions[(target_name, model_name)] = pred_log

model_benchmark_results = pd.DataFrame(benchmark_rows)
metric_cols = ["mae_original", "rmse_original", "mae_log", "r2_log"]
model_benchmark_results[metric_cols] = model_benchmark_results[metric_cols].round(4)
model_benchmark_results = model_benchmark_results.sort_values(["target", "mae_log", "mae_original"]).reset_index(drop=True)

model_benchmark_winners = (
    model_benchmark_results
    .sort_values(["target", "mae_log", "mae_original"])
    .groupby("target", as_index=False)
    .first()
    [["target", "model", "mae_original", "rmse_original", "mae_log", "r2_log"]]
)

benchmark_split_summary = pd.DataFrame([
    {
        "target": target_name,
        "train_rows": split["train_rows"],
        "valid_rows": split["valid_rows"],
        "n_categorical_features": len(split["categorical_features"]),
        "n_numeric_features": len(split["numeric_features"]),
        "categorical_features": ", ".join(split["categorical_features"]),
        "numeric_features": ", ".join(split["numeric_features"]),
    }
    for target_name, split in benchmark_splits.items()
])

print(f"Benchmark test size: {BENCHMARK_TEST_SIZE:.0%}; random_state={BENCHMARK_RANDOM_STATE}")
print(f"RandomForest benchmark enabled: {RUN_RANDOM_FOREST_BENCHMARK}")
display(benchmark_split_summary)

display(model_benchmark_results)

print("Best model per target by validation MAE on log1p(target):")
display(model_benchmark_winners)


Benchmark test size: 20%; random_state=42
RandomForest benchmark enabled: False


,target,train_rows,valid_rows,n_categorical_features,n_numeric_features,categorical_features,numeric_features
0,duration,16505,4127,12,10,"phase, therapeutic_area, sponsor_tier, therapeutic_modality, administration_complexity, endpoint_rigor, endpoint_structure, allocation, masking, intervention_model, primary_purpose, enrollment_type","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, enrollment_num, site_count_num, country_count_num, endpoint_duration_months_num, is_duration_unknown"
1,enrollment,16346,4087,12,9,"phase, therapeutic_area, sponsor_tier, therapeutic_modality, administration_complexity, endpoint_rigor, endpoint_structure, allocation, masking, intervention_model, primary_purpose, enrollment_type","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, site_count_num, country_count_num, endpoint_duration_months_num, is_duration_unknown"
2,site_count,15904,3976,12,9,"phase, therapeutic_area, sponsor_tier, therapeutic_modality, administration_complexity, endpoint_rigor, endpoint_structure, allocation, masking, intervention_model, primary_purpose, enrollment_type","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, enrollment_num, country_count_num, endpoint_duration_months_num, is_duration_unknown"
3,country_count,15903,3976,12,9,"phase, therapeutic_area, sponsor_tier, therapeutic_modality, administration_complexity, endpoint_rigor, endpoint_structure, allocation, masking, intervention_model, primary_purpose, enrollment_type","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, enrollment_num, site_count_num, endpoint_duration_months_num, is_duration_unknown"


,target,model,train_rows,valid_rows,n_features,mae_original,rmse_original,mae_log,r2_log
0,country_count,hist_gradient_boosting,15903,3976,21,1.8066,3.2268,0.3010,0.7187
1,country_count,xgboost,15903,3976,21,1.8262,3.2394,0.3060,0.7180
2,country_count,grouped_median_baseline,15903,3976,21,3.3302,6.5830,0.5603,-0.1819
3,duration,hist_gradient_boosting,16505,4127,22,9.7867,15.2742,0.3801,0.6010
4,duration,xgboost,16505,4127,22,9.8992,15.3092,0.3860,0.5939
5,duration,grouped_median_baseline,16505,4127,22,13.2857,19.7990,0.5221,0.2626
6,enrollment,hist_gradient_boosting,16346,4087,21,227.0085,1854.9309,0.5036,0.7332
7,enrollment,xgboost,16346,4087,21,226.1174,1823.8908,0.5108,0.7261
8,enrollment,grouped_median_baseline,16346,4087,21,328.2292,2078.0725,0.8351,0.3263
9,site_count,hist_gradient_boosting,15904,3976,21,14.5065,30.1161,0.5893,0.7146


Best model per target by validation MAE on log1p(target):


,target,model,mae_original,rmse_original,mae_log,r2_log
0,country_count,hist_gradient_boosting,1.8066,3.2268,0.3010,0.7187
1,duration,hist_gradient_boosting,9.7867,15.2742,0.3801,0.6010
2,enrollment,hist_gradient_boosting,227.0085,1854.9309,0.5036,0.7332
3,site_count,hist_gradient_boosting,14.5065,30.1161,0.5893,0.7146


#### <REF:MODEL_BENCHMARK_ENHANCED>

> #### **14. Enhanced Model Benchmark Run**
> Improve the first-pass benchmark with richer non-outcome features and train-only encoders. This block keeps the same completed-only training datasets and the same holdout split, but adds:
>
> - indication and sponsor context with target encoding fitted inside the training fold only;
> - market/disease-burden numeric features;
> - start-date trend features;
> - stronger tuned HGB/XGB candidates.
>
> These are still review benchmarks only. No fitted models or artifacts are saved.


In [ ]:
# <REF:MODEL_BENCHMARK_ENHANCED_CODE>
if "training_datasets" not in globals():
    raise RuntimeError("Run <REF:TRAINING_DATASETS_CODE> before enhanced model benchmarking.")
if "df_targets_base" not in globals():
    raise RuntimeError("Run <REF:TARGET_BUILD_CODE> before enhanced model benchmarking.")

from sklearn.preprocessing import TargetEncoder

ENHANCED_CONTEXT_COLUMNS = [
    "gbd_indication_name",
    "gbd_indication_name_3",
    "lead_sponsor_canonical",
    "daly_global",
    "daly_high_income",
    "chronic_ratio_global",
    "chronic_ratio_high_income",
    "market_skew_index",
]
ENHANCED_LOW_CARDINALITY_FEATURES = [
    "phase",
    "therapeutic_area",
    "sponsor_tier",
    "therapeutic_modality",
    "administration_complexity",
    "endpoint_rigor",
    "endpoint_structure",
    "allocation",
    "masking",
    "intervention_model",
    "primary_purpose",
    "enrollment_type",
]
ENHANCED_HIGH_CARDINALITY_FEATURES = [
    "gbd_indication_name",
    "gbd_indication_name_3",
    "lead_sponsor_canonical",
]
ENHANCED_NUMERIC_FEATURES = [
    "gbd_cause_id_3",
    "is_rare_disease",
    "has_placebo",
    "has_dmc",
    "number_of_arms_num",
    "enrollment_num",
    "site_count_num",
    "country_count_num",
    "endpoint_duration_months_num",
    "is_duration_unknown",
    "daly_global",
    "daly_high_income",
    "chronic_ratio_global",
    "chronic_ratio_high_income",
    "market_skew_index",
    "start_year",
    "start_month",
    "start_quarter",
    "start_epoch_days",
]

def add_enhanced_features(data):
    out = data.copy()
    add_cols = [
        c for c in ENHANCED_CONTEXT_COLUMNS
        if c in df_targets_base.columns and c not in out.columns
    ]
    if add_cols:
        out = out.merge(df_targets_base[["nct_id"] + add_cols], on="nct_id", how="left")

    out["start_date_parsed"] = pd.to_datetime(out["start_date_parsed"], errors="coerce")
    out["start_year"] = out["start_date_parsed"].dt.year
    out["start_month"] = out["start_date_parsed"].dt.month
    out["start_quarter"] = out["start_date_parsed"].dt.quarter
    out["start_epoch_days"] = (out["start_date_parsed"] - pd.Timestamp("2000-01-01")).dt.days
    return out

def get_enhanced_features_for_target(target_name, data):
    excluded = set(TARGET_FEATURE_EXCLUSIONS.get(target_name, []))
    low_card = [c for c in ENHANCED_LOW_CARDINALITY_FEATURES if c in data.columns and c not in excluded]
    high_card = [c for c in ENHANCED_HIGH_CARDINALITY_FEATURES if c in data.columns and c not in excluded]
    numeric = [c for c in ENHANCED_NUMERIC_FEATURES if c in data.columns and c not in excluded]
    return low_card, high_card, numeric

def make_enhanced_preprocessor(low_cardinality_features, high_cardinality_features, numeric_features):
    transformers = []
    if low_cardinality_features:
        transformers.append((
            "low_card_cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            low_cardinality_features,
        ))
    if high_cardinality_features:
        transformers.append((
            "high_card_cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                ("target_encoder", TargetEncoder(target_type="continuous", smooth="auto", random_state=BENCHMARK_RANDOM_STATE)),
            ]),
            high_cardinality_features,
        ))
    if numeric_features:
        transformers.append((
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ))
    return ColumnTransformer(transformers=transformers, remainder="drop")

ENHANCED_MODEL_CONFIG = {
    "enhanced_hgb": {
        "estimator": HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.035,
            max_iter=800,
            l2_regularization=0.03,
            max_leaf_nodes=31,
            random_state=BENCHMARK_RANDOM_STATE,
        ),
    },
    "enhanced_hgb_wide": {
        "estimator": HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.025,
            max_iter=1000,
            l2_regularization=0.01,
            max_leaf_nodes=63,
            random_state=BENCHMARK_RANDOM_STATE,
        ),
    },
}

if HAS_XGBOOST:
    ENHANCED_MODEL_CONFIG.update({
        "enhanced_xgb_depth3": {
            "estimator": XGBRegressor(
                objective="reg:squarederror",
                n_estimators=900,
                learning_rate=0.025,
                max_depth=3,
                min_child_weight=8,
                subsample=0.90,
                colsample_bytree=0.90,
                reg_lambda=2.0,
                random_state=BENCHMARK_RANDOM_STATE,
                n_jobs=-1,
            ),
        },
        "enhanced_xgb_depth4": {
            "estimator": XGBRegressor(
                objective="reg:squarederror",
                n_estimators=700,
                learning_rate=0.030,
                max_depth=4,
                min_child_weight=6,
                subsample=0.90,
                colsample_bytree=0.85,
                reg_lambda=2.0,
                random_state=BENCHMARK_RANDOM_STATE,
                n_jobs=-1,
            ),
        },
    })

enhanced_rows = []
enhanced_splits = {}

for target_name, data in training_datasets.items():
    target_data = add_enhanced_features(data)
    low_card_features, high_card_features, numeric_features = get_enhanced_features_for_target(target_name, target_data)
    feature_cols = low_card_features + high_card_features + numeric_features

    train_df, valid_df = train_test_split(
        target_data,
        test_size=BENCHMARK_TEST_SIZE,
        random_state=BENCHMARK_RANDOM_STATE,
    )
    y_train_log = train_df["target_value_log1p"].astype(float)
    y_valid_log = valid_df["target_value_log1p"].astype(float)

    enhanced_splits[target_name] = {
        "train_rows": len(train_df),
        "valid_rows": len(valid_df),
        "low_cardinality_features": low_card_features,
        "high_cardinality_features": high_card_features,
        "numeric_features": numeric_features,
    }

    for model_name, config in ENHANCED_MODEL_CONFIG.items():
        pipeline = Pipeline([
            ("preprocessor", make_enhanced_preprocessor(low_card_features, high_card_features, numeric_features)),
            ("model", config["estimator"]),
        ])
        pipeline.fit(train_df[feature_cols], y_train_log)
        pred_log = pipeline.predict(valid_df[feature_cols])
        metrics = evaluate_log_predictions(y_valid_log, pred_log)
        enhanced_rows.append({
            "target": target_name,
            "model": model_name,
            "train_rows": len(train_df),
            "valid_rows": len(valid_df),
            "n_raw_features": len(feature_cols),
            **metrics,
        })

enhanced_model_benchmark_results = pd.DataFrame(enhanced_rows)
metric_cols = ["mae_original", "rmse_original", "mae_log", "r2_log"]
enhanced_model_benchmark_results[metric_cols] = enhanced_model_benchmark_results[metric_cols].round(4)
enhanced_model_benchmark_results = enhanced_model_benchmark_results.sort_values(
    ["target", "mae_log", "mae_original"]
).reset_index(drop=True)

enhanced_model_benchmark_winners = (
    enhanced_model_benchmark_results
    .sort_values(["target", "mae_log", "mae_original"])
    .groupby("target", as_index=False)
    .first()
    [["target", "model", "mae_original", "rmse_original", "mae_log", "r2_log"]]
)

baseline_best = model_benchmark_winners.rename(columns={
    "model": "baseline_best_model",
    "mae_original": "baseline_mae_original",
    "rmse_original": "baseline_rmse_original",
    "mae_log": "baseline_mae_log",
    "r2_log": "baseline_r2_log",
})
enhanced_best = enhanced_model_benchmark_winners.rename(columns={
    "model": "enhanced_best_model",
    "mae_original": "enhanced_mae_original",
    "rmse_original": "enhanced_rmse_original",
    "mae_log": "enhanced_mae_log",
    "r2_log": "enhanced_r2_log",
})
enhanced_vs_baseline = baseline_best.merge(enhanced_best, on="target", how="inner")
enhanced_vs_baseline["mae_log_delta"] = (
    enhanced_vs_baseline["enhanced_mae_log"] - enhanced_vs_baseline["baseline_mae_log"]
).round(4)
enhanced_vs_baseline["mae_log_pct_improvement"] = (
    1 - enhanced_vs_baseline["enhanced_mae_log"] / enhanced_vs_baseline["baseline_mae_log"]
).round(4)
enhanced_vs_baseline["r2_log_delta"] = (
    enhanced_vs_baseline["enhanced_r2_log"] - enhanced_vs_baseline["baseline_r2_log"]
).round(4)

enhanced_split_summary = pd.DataFrame([
    {
        "target": target_name,
        "train_rows": split["train_rows"],
        "valid_rows": split["valid_rows"],
        "n_low_cardinality_features": len(split["low_cardinality_features"]),
        "n_high_cardinality_features": len(split["high_cardinality_features"]),
        "n_numeric_features": len(split["numeric_features"]),
        "high_cardinality_features": ", ".join(split["high_cardinality_features"]),
        "numeric_features": ", ".join(split["numeric_features"]),
    }
    for target_name, split in enhanced_splits.items()
])

print("Enhanced benchmark uses the same completed-only target datasets and 20% holdout split.")
print("High-cardinality target encoders are fitted inside each training fold only.")
display(enhanced_split_summary)

display(enhanced_model_benchmark_results)

print("Best enhanced model per target by validation MAE on log1p(target):")
display(enhanced_model_benchmark_winners)

print("Enhanced-vs-original benchmark delta:")
display(enhanced_vs_baseline)


Enhanced benchmark uses the same completed-only target datasets and 20% holdout split.
High-cardinality target encoders are fitted inside each training fold only.


,target,train_rows,valid_rows,n_low_cardinality_features,n_high_cardinality_features,n_numeric_features,high_cardinality_features,numeric_features
0,duration,16505,4127,12,3,19,"gbd_indication_name, gbd_indication_name_3, lead_sponsor_canonical","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, enrollment_num, site_count_num, country_count_num, endpoint_duration_months_num, is_duration_unknown, daly_global, daly_high_income, chronic_ratio_global, chronic_ratio_high_income, market_skew_index, start_year, start_month, start_quarter, start_epoch_days"
1,enrollment,16346,4087,12,3,18,"gbd_indication_name, gbd_indication_name_3, lead_sponsor_canonical","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, site_count_num, country_count_num, endpoint_duration_months_num, is_duration_unknown, daly_global, daly_high_income, chronic_ratio_global, chronic_ratio_high_income, market_skew_index, start_year, start_month, start_quarter, start_epoch_days"
2,site_count,15904,3976,12,3,18,"gbd_indication_name, gbd_indication_name_3, lead_sponsor_canonical","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, enrollment_num, country_count_num, endpoint_duration_months_num, is_duration_unknown, daly_global, daly_high_income, chronic_ratio_global, chronic_ratio_high_income, market_skew_index, start_year, start_month, start_quarter, start_epoch_days"
3,country_count,15903,3976,12,3,18,"gbd_indication_name, gbd_indication_name_3, lead_sponsor_canonical","gbd_cause_id_3, is_rare_disease, has_placebo, has_dmc, number_of_arms_num, enrollment_num, site_count_num, endpoint_duration_months_num, is_duration_unknown, daly_global, daly_high_income, chronic_ratio_global, chronic_ratio_high_income, market_skew_index, start_year, start_month, start_quarter, start_epoch_days"


,target,model,train_rows,valid_rows,n_raw_features,mae_original,rmse_original,mae_log,r2_log
0,country_count,enhanced_hgb_wide,15903,3976,33,1.7347,3.1044,0.2856,0.7399
1,country_count,enhanced_hgb,15903,3976,33,1.7412,3.1009,0.2885,0.7372
2,country_count,enhanced_xgb_depth4,15903,3976,33,1.7622,3.1401,0.2938,0.7348
3,country_count,enhanced_xgb_depth3,15903,3976,33,1.7892,3.1763,0.2987,0.7289
4,duration,enhanced_hgb_wide,16505,4127,34,9.4080,14.6486,0.3654,0.6269
5,duration,enhanced_hgb,16505,4127,34,9.4755,14.6991,0.3687,0.6232
6,duration,enhanced_xgb_depth4,16505,4127,34,9.5850,14.8090,0.3733,0.6182
7,duration,enhanced_xgb_depth3,16505,4127,34,9.6902,15.0007,0.3777,0.6094
8,enrollment,enhanced_hgb_wide,16346,4087,33,222.6836,1840.8220,0.4938,0.7427
9,enrollment,enhanced_hgb,16346,4087,33,225.8540,1856.7361,0.4973,0.7397


Best enhanced model per target by validation MAE on log1p(target):


,target,model,mae_original,rmse_original,mae_log,r2_log
0,country_count,enhanced_hgb_wide,1.7347,3.1044,0.2856,0.7399
1,duration,enhanced_hgb_wide,9.4080,14.6486,0.3654,0.6269
2,enrollment,enhanced_hgb_wide,222.6836,1840.8220,0.4938,0.7427
3,site_count,enhanced_hgb_wide,13.5657,29.9564,0.5182,0.7690


Enhanced-vs-original benchmark delta:


,target,baseline_best_model,baseline_mae_original,baseline_rmse_original,baseline_mae_log,baseline_r2_log,enhanced_best_model,enhanced_mae_original,enhanced_rmse_original,enhanced_mae_log,enhanced_r2_log,mae_log_delta,mae_log_pct_improvement,r2_log_delta
0,country_count,hist_gradient_boosting,1.8066,3.2268,0.3010,0.7187,enhanced_hgb_wide,1.7347,3.1044,0.2856,0.7399,-0.0154,0.0512,0.0212
1,duration,hist_gradient_boosting,9.7867,15.2742,0.3801,0.6010,enhanced_hgb_wide,9.4080,14.6486,0.3654,0.6269,-0.0147,0.0387,0.0259
2,enrollment,hist_gradient_boosting,227.0085,1854.9309,0.5036,0.7332,enhanced_hgb_wide,222.6836,1840.8220,0.4938,0.7427,-0.0098,0.0195,0.0095
3,site_count,hist_gradient_boosting,14.5065,30.1161,0.5893,0.7146,enhanced_hgb_wide,13.5657,29.9564,0.5182,0.7690,-0.0711,0.1207,0.0544


#### <REF:MODEL_BENCHMARK_REFINEMENT_EXPLAINABILITY>

> #### **15. Model Refinement and Feature Impact**
> Make one additional score-improvement attempt using validation-fold blends of the enhanced candidates, then explain the selected per-target predictor with model-agnostic permutation impact on the holdout set.
>
> Feature impact is reported as the increase in validation `MAE log1p(target)` when a raw input feature is shuffled. Larger values mean the selected predictor depends more on that feature. This is computed on the validation fold only and does not save models.


In [ ]:
# <REF:MODEL_BENCHMARK_REFINEMENT_EXPLAINABILITY_CODE>
if "ENHANCED_MODEL_CONFIG" not in globals():
    raise RuntimeError("Run <REF:MODEL_BENCHMARK_ENHANCED_CODE> before refinement and explainability.")

from sklearn.base import clone
import itertools

REFINEMENT_BLEND_WEIGHT_SETS = {
    2: [(0.25, 0.75), (0.40, 0.60), (0.50, 0.50), (0.60, 0.40), (0.75, 0.25)],
    3: [(1/3, 1/3, 1/3), (0.50, 0.25, 0.25), (0.25, 0.50, 0.25), (0.25, 0.25, 0.50)],
    4: [(0.25, 0.25, 0.25, 0.25)],
}
PERMUTATION_REPEATS = 5
PERMUTATION_RANDOM_STATE = 42

def make_fresh_enhanced_pipeline(model_name, low_card_features, high_card_features, numeric_features):
    return Pipeline([
        ("preprocessor", make_enhanced_preprocessor(low_card_features, high_card_features, numeric_features)),
        ("model", clone(ENHANCED_MODEL_CONFIG[model_name]["estimator"])),
    ])

def weighted_prediction(pipelines, weights, frame, feature_cols):
    pred = np.zeros(len(frame), dtype=float)
    for model_name, weight in zip(pipelines.keys(), weights):
        pred += weight * pipelines[model_name].predict(frame[feature_cols])
    return pred

def format_blend_name(model_names, weights):
    parts = [f"{name}:{weight:.2f}" for name, weight in zip(model_names, weights)]
    return "blend__" + "__".join(parts)

refinement_rows = []
selected_model_specs = {}
selected_model_context = {}

for target_name, data in training_datasets.items():
    target_data = add_enhanced_features(data)
    low_card_features, high_card_features, numeric_features = get_enhanced_features_for_target(target_name, target_data)
    feature_cols = low_card_features + high_card_features + numeric_features

    train_df, valid_df = train_test_split(
        target_data,
        test_size=BENCHMARK_TEST_SIZE,
        random_state=BENCHMARK_RANDOM_STATE,
    )
    y_train_log = train_df["target_value_log1p"].astype(float)
    y_valid_log = valid_df["target_value_log1p"].astype(float)

    fitted_pipelines = {}
    validation_predictions = {}
    for model_name in ENHANCED_MODEL_CONFIG:
        pipeline = make_fresh_enhanced_pipeline(model_name, low_card_features, high_card_features, numeric_features)
        pipeline.fit(train_df[feature_cols], y_train_log)
        fitted_pipelines[model_name] = pipeline
        validation_predictions[model_name] = pipeline.predict(valid_df[feature_cols])

        metrics = evaluate_log_predictions(y_valid_log, validation_predictions[model_name])
        refinement_rows.append({
            "target": target_name,
            "model": model_name,
            "kind": "single",
            "weights": model_name,
            **metrics,
        })

    model_names = list(validation_predictions)
    for combo_size, weight_sets in REFINEMENT_BLEND_WEIGHT_SETS.items():
        if combo_size > len(model_names):
            continue
        for combo in itertools.combinations(model_names, combo_size):
            combo_preds = [validation_predictions[name] for name in combo]
            for weights in weight_sets:
                pred_log = np.sum([weight * pred for weight, pred in zip(weights, combo_preds)], axis=0)
                metrics = evaluate_log_predictions(y_valid_log, pred_log)
                refinement_rows.append({
                    "target": target_name,
                    "model": format_blend_name(combo, weights),
                    "kind": "blend",
                    "weights": ", ".join(f"{name}={weight:.2f}" for name, weight in zip(combo, weights)),
                    **metrics,
                })

    target_refinement = pd.DataFrame([r for r in refinement_rows if r["target"] == target_name])
    best_row = target_refinement.sort_values(["mae_log", "mae_original"]).iloc[0].to_dict()
    if best_row["kind"] == "single":
        selected_names = [best_row["model"]]
        selected_weights = [1.0]
    else:
        selected_names = []
        selected_weights = []
        for item in best_row["weights"].split(", "):
            name, weight = item.rsplit("=", 1)
            selected_names.append(name)
            selected_weights.append(float(weight))

    selected_model_specs[target_name] = {
        "kind": best_row["kind"],
        "model": best_row["model"],
        "weights": best_row["weights"],
        "selected_names": selected_names,
        "selected_weights": selected_weights,
    }
    selected_model_context[target_name] = {
        "train_df": train_df,
        "valid_df": valid_df,
        "y_valid_log": y_valid_log,
        "feature_cols": feature_cols,
        "low_card_features": low_card_features,
        "high_card_features": high_card_features,
        "numeric_features": numeric_features,
        "pipelines": {name: fitted_pipelines[name] for name in selected_names},
    }

refined_model_benchmark_results = pd.DataFrame(refinement_rows)
metric_cols = ["mae_original", "rmse_original", "mae_log", "r2_log"]
refined_model_benchmark_results[metric_cols] = refined_model_benchmark_results[metric_cols].round(4)
refined_model_benchmark_results = refined_model_benchmark_results.sort_values(
    ["target", "mae_log", "mae_original"]
).reset_index(drop=True)

refined_model_benchmark_winners = (
    refined_model_benchmark_results
    .groupby("target", as_index=False)
    .first()
    [["target", "model", "kind", "weights", "mae_original", "rmse_original", "mae_log", "r2_log"]]
)

enhanced_best_for_refinement = enhanced_model_benchmark_winners.rename(columns={
    "model": "enhanced_best_model",
    "mae_original": "enhanced_mae_original",
    "rmse_original": "enhanced_rmse_original",
    "mae_log": "enhanced_mae_log",
    "r2_log": "enhanced_r2_log",
})
refined_best_for_comparison = refined_model_benchmark_winners.rename(columns={
    "model": "refined_best_model",
    "mae_original": "refined_mae_original",
    "rmse_original": "refined_rmse_original",
    "mae_log": "refined_mae_log",
    "r2_log": "refined_r2_log",
})
refined_vs_enhanced = enhanced_best_for_refinement.merge(refined_best_for_comparison, on="target", how="inner")
refined_vs_enhanced["mae_log_delta"] = (
    refined_vs_enhanced["refined_mae_log"] - refined_vs_enhanced["enhanced_mae_log"]
).round(4)
refined_vs_enhanced["mae_log_pct_improvement"] = (
    1 - refined_vs_enhanced["refined_mae_log"] / refined_vs_enhanced["enhanced_mae_log"]
).round(4)
refined_vs_enhanced["r2_log_delta"] = (
    refined_vs_enhanced["refined_r2_log"] - refined_vs_enhanced["enhanced_r2_log"]
).round(4)

permutation_rows = []
rng = np.random.default_rng(PERMUTATION_RANDOM_STATE)
for target_name, context in selected_model_context.items():
    spec = selected_model_specs[target_name]
    valid_df = context["valid_df"]
    y_valid_log = context["y_valid_log"]
    feature_cols = context["feature_cols"]
    selected_pipelines = context["pipelines"]
    selected_weights = spec["selected_weights"]

    baseline_pred = weighted_prediction(selected_pipelines, selected_weights, valid_df, feature_cols)
    baseline_mae_log = mean_absolute_error(y_valid_log, baseline_pred)

    for feature in feature_cols:
        deltas = []
        for _ in range(PERMUTATION_REPEATS):
            permuted = valid_df[feature_cols].copy()
            permuted[feature] = rng.permutation(permuted[feature].to_numpy())
            permuted_pred = weighted_prediction(selected_pipelines, selected_weights, permuted, feature_cols)
            permuted_mae_log = mean_absolute_error(y_valid_log, permuted_pred)
            deltas.append(permuted_mae_log - baseline_mae_log)

        mean_delta = float(np.mean(deltas))
        permutation_rows.append({
            "target": target_name,
            "selected_model": spec["model"],
            "feature": feature,
            "baseline_mae_log": baseline_mae_log,
            "mae_log_increase_when_shuffled": mean_delta,
            "relative_mae_log_increase": mean_delta / baseline_mae_log if baseline_mae_log else np.nan,
        })

feature_impact_results = pd.DataFrame(permutation_rows)
feature_impact_results["baseline_mae_log"] = feature_impact_results["baseline_mae_log"].round(4)
feature_impact_results["mae_log_increase_when_shuffled"] = feature_impact_results["mae_log_increase_when_shuffled"].round(4)
feature_impact_results["relative_mae_log_increase"] = feature_impact_results["relative_mae_log_increase"].round(4)
feature_impact_results = feature_impact_results.sort_values(
    ["target", "mae_log_increase_when_shuffled"], ascending=[True, False]
).reset_index(drop=True)

top_feature_impact = feature_impact_results.groupby("target", as_index=False).head(5).reset_index(drop=True)

print("Refinement candidates include singles and validation-fold blends of enhanced models.")
display(refined_model_benchmark_results.groupby("target", as_index=False).head(8).reset_index(drop=True))

print("Selected refined predictor per target by validation MAE on log1p(target):")
display(refined_model_benchmark_winners)

print("Refined-vs-enhanced benchmark delta:")
display(refined_vs_enhanced)

print(f"Top 5 raw feature impacts per selected target predictor; permutation repeats={PERMUTATION_REPEATS}.")
display(top_feature_impact)


Refinement candidates include singles and validation-fold blends of enhanced models.


,target,model,kind,weights,mae_original,rmse_original,mae_log,r2_log
0,country_count,enhanced_hgb_wide,single,enhanced_hgb_wide,1.7347,3.1044,0.2856,0.7399
1,country_count,blend__enhanced_hgb:0.25__enhanced_hgb_wide:0.75,blend,"enhanced_hgb=0.25, enhanced_hgb_wide=0.75",1.7333,3.0981,0.2858,0.7402
2,country_count,blend__enhanced_hgb:0.40__enhanced_hgb_wide:0.60,blend,"enhanced_hgb=0.40, enhanced_hgb_wide=0.60",1.7333,3.0961,0.2861,0.7401
3,country_count,blend__enhanced_hgb:0.50__enhanced_hgb_wide:0.50,blend,"enhanced_hgb=0.50, enhanced_hgb_wide=0.50",1.7337,3.0954,0.2864,0.7399
4,country_count,blend__enhanced_hgb_wide:0.75__enhanced_xgb_depth4:0.25,blend,"enhanced_hgb_wide=0.75, enhanced_xgb_depth4=0.25",1.7344,3.0968,0.2865,0.7411
5,country_count,blend__enhanced_hgb:0.60__enhanced_hgb_wide:0.40,blend,"enhanced_hgb=0.60, enhanced_hgb_wide=0.40",1.7344,3.0954,0.2867,0.7396
6,country_count,blend__enhanced_hgb:0.25__enhanced_hgb_wide:0.50__enhanced_xgb_depth4:0.25,blend,"enhanced_hgb=0.25, enhanced_hgb_wide=0.50, enhanced_xgb_depth4=0.25",1.7345,3.0941,0.2870,0.7408
7,country_count,blend__enhanced_hgb:0.75__enhanced_hgb_wide:0.25,blend,"enhanced_hgb=0.75, enhanced_hgb_wide=0.25",1.7362,3.0964,0.2873,0.7389
8,duration,enhanced_hgb_wide,single,enhanced_hgb_wide,9.4080,14.6486,0.3654,0.6269
9,duration,blend__enhanced_hgb:0.25__enhanced_hgb_wide:0.75,blend,"enhanced_hgb=0.25, enhanced_hgb_wide=0.75",9.4079,14.6440,0.3655,0.6274


Selected refined predictor per target by validation MAE on log1p(target):


,target,model,kind,weights,mae_original,rmse_original,mae_log,r2_log
0,country_count,enhanced_hgb_wide,single,enhanced_hgb_wide,1.7347,3.1044,0.2856,0.7399
1,duration,enhanced_hgb_wide,single,enhanced_hgb_wide,9.4080,14.6486,0.3654,0.6269
2,enrollment,blend__enhanced_hgb_wide:0.75__enhanced_xgb_depth4:0.25,blend,"enhanced_hgb_wide=0.75, enhanced_xgb_depth4=0.25",223.2017,1842.8890,0.4925,0.7439
3,site_count,blend__enhanced_hgb_wide:0.75__enhanced_xgb_depth4:0.25,blend,"enhanced_hgb_wide=0.75, enhanced_xgb_depth4=0.25",13.5343,30.1097,0.5181,0.7698


Refined-vs-enhanced benchmark delta:


,target,enhanced_best_model,enhanced_mae_original,enhanced_rmse_original,enhanced_mae_log,enhanced_r2_log,refined_best_model,kind,weights,refined_mae_original,refined_rmse_original,refined_mae_log,refined_r2_log,mae_log_delta,mae_log_pct_improvement,r2_log_delta
0,country_count,enhanced_hgb_wide,1.7347,3.1044,0.2856,0.7399,enhanced_hgb_wide,single,enhanced_hgb_wide,1.7347,3.1044,0.2856,0.7399,0.0000,0.0000,0.0000
1,duration,enhanced_hgb_wide,9.4080,14.6486,0.3654,0.6269,enhanced_hgb_wide,single,enhanced_hgb_wide,9.4080,14.6486,0.3654,0.6269,0.0000,0.0000,0.0000
2,enrollment,enhanced_hgb_wide,222.6836,1840.8220,0.4938,0.7427,blend__enhanced_hgb_wide:0.75__enhanced_xgb_depth4:0.25,blend,"enhanced_hgb_wide=0.75, enhanced_xgb_depth4=0.25",223.2017,1842.8890,0.4925,0.7439,-0.0013,0.0026,0.0012
3,site_count,enhanced_hgb_wide,13.5657,29.9564,0.5182,0.7690,blend__enhanced_hgb_wide:0.75__enhanced_xgb_depth4:0.25,blend,"enhanced_hgb_wide=0.75, enhanced_xgb_depth4=0.25",13.5343,30.1097,0.5181,0.7698,-0.0001,0.0002,0.0008


Top 5 raw feature impacts per selected target predictor; permutation repeats=5.


,target,selected_model,feature,baseline_mae_log,mae_log_increase_when_shuffled,relative_mae_log_increase
0,country_count,enhanced_hgb_wide,site_count_num,0.2856,0.3864,1.3529
1,country_count,enhanced_hgb_wide,lead_sponsor_canonical,0.2856,0.0508,0.1778
2,country_count,enhanced_hgb_wide,is_rare_disease,0.2856,0.0119,0.0418
3,country_count,enhanced_hgb_wide,has_dmc,0.2856,0.0094,0.0329
4,country_count,enhanced_hgb_wide,enrollment_num,0.2856,0.0072,0.0252
5,duration,enhanced_hgb_wide,endpoint_duration_months_num,0.3654,0.1354,0.3705
6,duration,enhanced_hgb_wide,gbd_indication_name,0.3654,0.0345,0.0944
7,duration,enhanced_hgb_wide,country_count_num,0.3654,0.0205,0.0562
8,duration,enhanced_hgb_wide,enrollment_num,0.3654,0.0187,0.0511
9,duration,enhanced_hgb_wide,lead_sponsor_canonical,0.3654,0.0170,0.0466


#### <REF:SEQUENCED_SIMULATION_BENCHMARK>

> #### **16. Sequenced Estimation Benchmark**
> Test realistic y-to-y sequencing without using true completed downstream/oracle operational values as model inputs.
>
> This block creates a common completed-only cohort with all four cleaned targets, then evaluates candidate directed orders. For each order, upstream predictions are generated as out-of-fold predictions on the training cohort and as fitted-model predictions on the validation cohort. Downstream models receive those predicted upstream values, not true target values.
>
> This is the credibility bridge between high-scoring oracle benchmarks and the deployable estimation flow.


In [ ]:
# <REF:SEQUENCED_SIMULATION_BENCHMARK_CODE>
if "ENHANCED_MODEL_CONFIG" not in globals():
    raise RuntimeError("Run <REF:MODEL_BENCHMARK_ENHANCED_CODE> before sequenced benchmarking.")

from sklearn.base import clone
from sklearn.model_selection import KFold

SEQUENCED_TARGETS = ["country_count", "site_count", "enrollment", "duration"]
SEQUENCED_PRED_COLUMNS = {target: f"seq_pred_{target}_log1p" for target in SEQUENCED_TARGETS}
SEQUENCED_BASE_NUMERIC_FEATURES = [
    c for c in ENHANCED_NUMERIC_FEATURES
    if c not in ["country_count_num", "site_count_num", "enrollment_num"]
]
SEQUENCED_ORDER_CANDIDATES = [
    ("enrollment", "site_count", "country_count", "duration"),
    ("country_count", "site_count", "enrollment", "duration"),
    ("enrollment", "country_count", "site_count", "duration"),
    ("site_count", "country_count", "enrollment", "duration"),
    ("country_count", "enrollment", "site_count", "duration"),
    ("site_count", "enrollment", "country_count", "duration"),
]
SEQUENCED_MODEL_NAME = "enhanced_hgb_wide"
SEQUENCED_OOF_SPLITS = 5

def build_common_sequenced_dataset():
    common = None
    for target_name in SEQUENCED_TARGETS:
        target_data = add_enhanced_features(training_datasets[target_name]).copy()
        target_values = target_data[["nct_id", "target_value", "target_value_log1p"]].rename(columns={
            "target_value": f"{target_name}_target_value",
            "target_value_log1p": f"{target_name}_target_log1p",
        })
        if common is None:
            common = target_data.rename(columns={
                "target_value": f"{target_name}_target_value",
                "target_value_log1p": f"{target_name}_target_log1p",
            })
        else:
            common = common.merge(target_values, on="nct_id", how="inner")
    return common

def get_sequenced_features_for_stage(stage_name, order, frame):
    low_card = [c for c in ENHANCED_LOW_CARDINALITY_FEATURES if c in frame.columns]
    high_card = [c for c in ENHANCED_HIGH_CARDINALITY_FEATURES if c in frame.columns]
    numeric = [c for c in SEQUENCED_BASE_NUMERIC_FEATURES if c in frame.columns]
    for upstream_target in order[:order.index(stage_name)]:
        pred_col = SEQUENCED_PRED_COLUMNS[upstream_target]
        if pred_col in frame.columns:
            numeric.append(pred_col)
    return low_card, high_card, numeric

def fit_predict_sequenced_stage(stage_name, order, train_df, valid_df):
    low_card_features, high_card_features, numeric_features = get_sequenced_features_for_stage(stage_name, order, train_df)
    feature_cols = low_card_features + high_card_features + numeric_features
    y_train_log = train_df[f"{stage_name}_target_log1p"].astype(float)

    pipeline = Pipeline([
        ("preprocessor", make_enhanced_preprocessor(low_card_features, high_card_features, numeric_features)),
        ("model", clone(ENHANCED_MODEL_CONFIG[SEQUENCED_MODEL_NAME]["estimator"])),
    ])
    pipeline.fit(train_df[feature_cols], y_train_log)
    pred_log = pipeline.predict(valid_df[feature_cols])
    return pred_log, feature_cols

def make_oof_sequenced_predictions(stage_name, order, train_df):
    pred_log = np.zeros(len(train_df), dtype=float)
    kfold = KFold(n_splits=SEQUENCED_OOF_SPLITS, shuffle=True, random_state=BENCHMARK_RANDOM_STATE)
    for fit_idx, hold_idx in kfold.split(train_df):
        fold_train = train_df.iloc[fit_idx].copy()
        fold_hold = train_df.iloc[hold_idx].copy()
        fold_pred, _ = fit_predict_sequenced_stage(stage_name, order, fold_train, fold_hold)
        pred_log[hold_idx] = fold_pred
    return pred_log

sequenced_common_df = build_common_sequenced_dataset()
sequenced_train_df, sequenced_valid_df = train_test_split(
    sequenced_common_df,
    test_size=BENCHMARK_TEST_SIZE,
    random_state=BENCHMARK_RANDOM_STATE,
)
sequenced_train_df = sequenced_train_df.reset_index(drop=True)
sequenced_valid_df = sequenced_valid_df.reset_index(drop=True)

sequenced_rows = []
for order in SEQUENCED_ORDER_CANDIDATES:
    train_work = sequenced_train_df.copy()
    valid_work = sequenced_valid_df.copy()
    train_stage_predictions = {}
    valid_stage_predictions = {}

    for stage_name in order:
        for upstream_target in order[:order.index(stage_name)]:
            pred_col = SEQUENCED_PRED_COLUMNS[upstream_target]
            train_work[pred_col] = train_stage_predictions[upstream_target]
            valid_work[pred_col] = valid_stage_predictions[upstream_target]

        valid_pred_log, feature_cols = fit_predict_sequenced_stage(stage_name, order, train_work, valid_work)
        y_valid_log = valid_work[f"{stage_name}_target_log1p"].astype(float)
        metrics = evaluate_log_predictions(y_valid_log, valid_pred_log)
        sequenced_rows.append({
            "order": " > ".join(order),
            "target": stage_name,
            "model": SEQUENCED_MODEL_NAME,
            "train_rows": len(train_work),
            "valid_rows": len(valid_work),
            "n_features": len(feature_cols),
            **metrics,
        })

        train_stage_predictions[stage_name] = make_oof_sequenced_predictions(stage_name, order, train_work)
        valid_stage_predictions[stage_name] = valid_pred_log

sequenced_benchmark_results = pd.DataFrame(sequenced_rows)
metric_cols = ["mae_original", "rmse_original", "mae_log", "r2_log"]
sequenced_benchmark_results[metric_cols] = sequenced_benchmark_results[metric_cols].round(4)
sequenced_order_summary = (
    sequenced_benchmark_results
    .groupby("order", as_index=False)
    .agg(
        mean_mae_log=("mae_log", "mean"),
        sum_mae_log=("mae_log", "sum"),
        mean_r2_log=("r2_log", "mean"),
    )
    .sort_values(["mean_mae_log", "sum_mae_log"])
    .reset_index(drop=True)
)
sequenced_order_summary[["mean_mae_log", "sum_mae_log", "mean_r2_log"]] = sequenced_order_summary[
    ["mean_mae_log", "sum_mae_log", "mean_r2_log"]
].round(4)

best_sequenced_order = sequenced_order_summary.iloc[0]["order"]
best_sequenced_order_results = (
    sequenced_benchmark_results
    .loc[sequenced_benchmark_results["order"].eq(best_sequenced_order)]
    .sort_values("target")
    .reset_index(drop=True)
)

print("Sequenced benchmark uses common completed-only rows with all four cleaned targets.")
print(f"Common cohort: {len(sequenced_common_df):,} rows; train={len(sequenced_train_df):,}; valid={len(sequenced_valid_df):,}")
print(f"Sequenced model per stage: {SEQUENCED_MODEL_NAME}; OOF folds={SEQUENCED_OOF_SPLITS}")
display(sequenced_order_summary)

print("Best sequenced order target-level results:")
display(best_sequenced_order_results)

print("All sequenced candidate results:")
display(sequenced_benchmark_results.sort_values(["order", "target"]).reset_index(drop=True))


Sequenced benchmark uses common completed-only rows with all four cleaned targets.
Common cohort: 19,521 rows; train=15,616; valid=3,905
Sequenced model per stage: enhanced_hgb_wide; OOF folds=5


,order,mean_mae_log,sum_mae_log,mean_r2_log
0,enrollment > site_count > country_count > duration,0.5359,2.1438,0.5804
1,country_count > site_count > enrollment > duration,0.5360,2.1441,0.5785
2,enrollment > country_count > site_count > duration,0.5374,2.1498,0.5776
3,site_count > country_count > enrollment > duration,0.5392,2.1568,0.5762
4,country_count > enrollment > site_count > duration,0.5392,2.1568,0.5773
5,site_count > enrollment > country_count > duration,0.5397,2.1588,0.5759


Best sequenced order target-level results:


,order,target,model,train_rows,valid_rows,n_features,mae_original,rmse_original,mae_log,r2_log,mean_mae_log_by_order
0,enrollment > site_count > country_count > duration,country_count,enhanced_hgb_wide,15616,3905,33,2.5824,4.7123,0.4202,0.5023,0.536
1,enrollment > site_count > country_count > duration,duration,enhanced_hgb_wide,15616,3905,34,9.2906,14.1194,0.3725,0.6124,0.536
2,enrollment > site_count > country_count > duration,enrollment,enhanced_hgb_wide,15616,3905,31,231.6184,1066.8654,0.6013,0.6402,0.536
3,enrollment > site_count > country_count > duration,site_count,enhanced_hgb_wide,15616,3905,32,21.4106,45.6220,0.7498,0.5665,0.536


All sequenced candidate results:


,order,target,model,train_rows,valid_rows,n_features,mae_original,rmse_original,mae_log,r2_log,mean_mae_log_by_order
0,country_count > enrollment > site_count > duration,country_count,enhanced_hgb_wide,15616,3905,31,2.5591,4.7108,0.4174,0.5054,0.5392
1,country_count > enrollment > site_count > duration,duration,enhanced_hgb_wide,15616,3905,34,9.3474,14.2260,0.3740,0.6080,0.5392
2,country_count > enrollment > site_count > duration,enrollment,enhanced_hgb_wide,15616,3905,32,235.3007,1081.3316,0.6036,0.6388,0.5392
3,country_count > enrollment > site_count > duration,site_count,enhanced_hgb_wide,15616,3905,33,21.8555,48.0244,0.7618,0.5569,0.5392
4,country_count > site_count > enrollment > duration,country_count,enhanced_hgb_wide,15616,3905,31,2.5591,4.7108,0.4174,0.5054,0.5360
5,country_count > site_count > enrollment > duration,duration,enhanced_hgb_wide,15616,3905,34,9.3386,14.2198,0.3743,0.6068,0.5360
6,country_count > site_count > enrollment > duration,enrollment,enhanced_hgb_wide,15616,3905,33,233.2799,1069.9294,0.6032,0.6380,0.5360
7,country_count > site_count > enrollment > duration,site_count,enhanced_hgb_wide,15616,3905,32,21.3737,46.9283,0.7492,0.5639,0.5360
8,enrollment > country_count > site_count > duration,country_count,enhanced_hgb_wide,15616,3905,32,2.5965,4.7325,0.4246,0.4966,0.5375
9,enrollment > country_count > site_count > duration,duration,enhanced_hgb_wide,15616,3905,34,9.3570,14.2417,0.3742,0.6088,0.5375


#### <REF:DEPENDENCY_PRUNED_AND_RECONCILIATION_BENCHMARK>

> #### **17. Dependency-Pruned Benchmark and Operational Reconciliation**
> Test the model family without any operational target fields that would need to be guessed first: `enrollment_num`, `site_count_num`, and `country_count_num`.
>
> This block answers: how well can we predict each y from design, sponsor, indication, endpoint, market, and calendar features only? It also creates a small reconciliation/audit function for independently predicted enrollment/site/country combinations, so implausible bundles such as very low patients across many sites are visible before implementation.


In [ ]:
# <REF:DEPENDENCY_PRUNED_AND_RECONCILIATION_BENCHMARK_CODE>
if "ENHANCED_MODEL_CONFIG" not in globals():
    raise RuntimeError("Run <REF:MODEL_BENCHMARK_ENHANCED_CODE> before dependency-pruned benchmarking.")

from sklearn.base import clone

GUESSED_OPERATIONAL_FEATURES = ["enrollment_num", "site_count_num", "country_count_num"]
DEPENDENCY_PRUNED_NUMERIC_FEATURES = [
    c for c in ENHANCED_NUMERIC_FEATURES
    if c not in GUESSED_OPERATIONAL_FEATURES
]

def get_dependency_pruned_features(data):
    low_card = [c for c in ENHANCED_LOW_CARDINALITY_FEATURES if c in data.columns]
    high_card = [c for c in ENHANCED_HIGH_CARDINALITY_FEATURES if c in data.columns]
    numeric = [c for c in DEPENDENCY_PRUNED_NUMERIC_FEATURES if c in data.columns]
    return low_card, high_card, numeric

def fit_dependency_pruned_model(model_name, target_name, train_df, valid_df):
    low_card_features, high_card_features, numeric_features = get_dependency_pruned_features(train_df)
    feature_cols = low_card_features + high_card_features + numeric_features
    y_train_log = train_df["target_value_log1p"].astype(float)
    pipeline = Pipeline([
        ("preprocessor", make_enhanced_preprocessor(low_card_features, high_card_features, numeric_features)),
        ("model", clone(ENHANCED_MODEL_CONFIG[model_name]["estimator"])),
    ])
    pipeline.fit(train_df[feature_cols], y_train_log)
    pred_log = pipeline.predict(valid_df[feature_cols])
    return pipeline, pred_log, feature_cols

pruned_rows = []
pruned_best_models = {}
pruned_prediction_frames = {}

for target_name, data in training_datasets.items():
    target_data = add_enhanced_features(data)
    train_df, valid_df = train_test_split(
        target_data,
        test_size=BENCHMARK_TEST_SIZE,
        random_state=BENCHMARK_RANDOM_STATE,
    )
    y_valid_log = valid_df["target_value_log1p"].astype(float)
    best = None

    for model_name in ENHANCED_MODEL_CONFIG:
        pipeline, pred_log, feature_cols = fit_dependency_pruned_model(model_name, target_name, train_df, valid_df)
        metrics = evaluate_log_predictions(y_valid_log, pred_log)
        row = {
            "target": target_name,
            "model": model_name,
            "train_rows": len(train_df),
            "valid_rows": len(valid_df),
            "n_features": len(feature_cols),
            **metrics,
        }
        pruned_rows.append(row)
        score = (metrics["mae_log"], metrics["mae_original"])
        if best is None or score < best["score"]:
            best = {
                "score": score,
                "model": model_name,
                "pipeline": pipeline,
                "pred_log": pred_log,
                "feature_cols": feature_cols,
                "train_df": train_df,
                "valid_df": valid_df,
                "metrics": metrics,
            }

    pruned_best_models[target_name] = best
    pruned_prediction_frames[target_name] = valid_df[["nct_id", "target_value", "target_value_log1p"]].copy().assign(
        target=target_name,
        pred_log=best["pred_log"],
        pred_value=np.maximum(np.expm1(best["pred_log"]), 0),
    )

dependency_pruned_results = pd.DataFrame(pruned_rows)
metric_cols = ["mae_original", "rmse_original", "mae_log", "r2_log"]
dependency_pruned_results[metric_cols] = dependency_pruned_results[metric_cols].round(4)
dependency_pruned_results = dependency_pruned_results.sort_values(["target", "mae_log", "mae_original"]).reset_index(drop=True)

dependency_pruned_winners = (
    dependency_pruned_results
    .groupby("target", as_index=False)
    .first()
    [["target", "model", "mae_original", "rmse_original", "mae_log", "r2_log", "n_features"]]
)

def make_prediction_bundle(prediction_frames):
    pieces = []
    for target_name, frame in prediction_frames.items():
        pieces.append(frame[["nct_id", "pred_value", "target_value"]].rename(columns={
            "pred_value": f"pred_{target_name}",
            "target_value": f"actual_{target_name}",
        }))
    bundle = pieces[0]
    for piece in pieces[1:]:
        bundle = bundle.merge(piece, on="nct_id", how="inner")
    return bundle

def add_operational_reconciliation_flags(bundle, ratio_bounds=None):
    out = bundle.copy()
    eps = 1e-9
    out["pred_patients_per_site"] = out["pred_enrollment"] / np.maximum(out["pred_site_count"], eps)
    out["pred_sites_per_country"] = out["pred_site_count"] / np.maximum(out["pred_country_count"], eps)
    out["pred_patients_per_country"] = out["pred_enrollment"] / np.maximum(out["pred_country_count"], eps)

    out["flag_below_min_enrollment"] = out["pred_enrollment"] < 5
    out["flag_below_min_site_count"] = out["pred_site_count"] < 1
    out["flag_below_min_country_count"] = out["pred_country_count"] < 1
    out["flag_more_countries_than_sites"] = out["pred_country_count"] > out["pred_site_count"]
    out["flag_less_than_one_patient_per_site"] = out["pred_patients_per_site"] < 1

    if ratio_bounds is not None:
        out["flag_patients_per_site_outside_train_range"] = ~out["pred_patients_per_site"].between(
            ratio_bounds["patients_per_site"][0], ratio_bounds["patients_per_site"][1]
        )
        out["flag_sites_per_country_outside_train_range"] = ~out["pred_sites_per_country"].between(
            ratio_bounds["sites_per_country"][0], ratio_bounds["sites_per_country"][1]
        )
        out["flag_patients_per_country_outside_train_range"] = ~out["pred_patients_per_country"].between(
            ratio_bounds["patients_per_country"][0], ratio_bounds["patients_per_country"][1]
        )

    flag_cols = [c for c in out.columns if c.startswith("flag_")]
    out["n_reconciliation_flags"] = out[flag_cols].sum(axis=1)
    return out

common_actual_bundle = make_prediction_bundle({
    target_name: pruned_best_models[target_name]["train_df"][["nct_id", "target_value"]].copy().assign(
        pred_value=pruned_best_models[target_name]["train_df"]["target_value"]
    )
    for target_name in ["enrollment", "site_count", "country_count"]
})
common_actual_bundle["actual_patients_per_site"] = common_actual_bundle["actual_enrollment"] / common_actual_bundle["actual_site_count"].clip(lower=1)
common_actual_bundle["actual_sites_per_country"] = common_actual_bundle["actual_site_count"] / common_actual_bundle["actual_country_count"].clip(lower=1)
common_actual_bundle["actual_patients_per_country"] = common_actual_bundle["actual_enrollment"] / common_actual_bundle["actual_country_count"].clip(lower=1)

ratio_bounds = {
    "patients_per_site": tuple(common_actual_bundle["actual_patients_per_site"].quantile([0.01, 0.99]).round(4)),
    "sites_per_country": tuple(common_actual_bundle["actual_sites_per_country"].quantile([0.01, 0.99]).round(4)),
    "patients_per_country": tuple(common_actual_bundle["actual_patients_per_country"].quantile([0.01, 0.99]).round(4)),
}

pruned_prediction_bundle = make_prediction_bundle({
    target_name: pruned_prediction_frames[target_name]
    for target_name in ["enrollment", "site_count", "country_count"]
})
pruned_reconciliation_audit = add_operational_reconciliation_flags(pruned_prediction_bundle, ratio_bounds=ratio_bounds)

reconciliation_flag_cols = [c for c in pruned_reconciliation_audit.columns if c.startswith("flag_")]
reconciliation_summary = (
    pruned_reconciliation_audit[reconciliation_flag_cols]
    .sum()
    .rename("n_flagged")
    .reset_index()
    .rename(columns={"index": "flag"})
)
reconciliation_summary["pct_flagged"] = (
    reconciliation_summary["n_flagged"] / len(pruned_reconciliation_audit)
).round(4)

worst_reconciliation_examples = (
    pruned_reconciliation_audit
    .sort_values([
        "n_reconciliation_flags",
        "pred_patients_per_site",
        "pred_sites_per_country",
    ], ascending=[False, True, False])
    .head(20)
)

print("Dependency-pruned benchmark excludes enrollment_num, site_count_num, and country_count_num for every target.")
print("Sponsor class is still included through sponsor_tier; lead_sponsor_canonical is target-encoded inside each training fold.")
display(dependency_pruned_results)

print("Best dependency-pruned model per target:")
display(dependency_pruned_winners)

print("Training-derived ratio bounds used for reconciliation audit:")
display(pd.DataFrame([
    {"ratio": ratio_name, "p01": bounds[0], "p99": bounds[1]}
    for ratio_name, bounds in ratio_bounds.items()
]))

print("Independent prediction reconciliation flags:")
display(reconciliation_summary)

print("Worst reconciliation examples from dependency-pruned independent predictions:")
display(worst_reconciliation_examples)


Dependency-pruned benchmark excludes enrollment_num, site_count_num, and country_count_num for every target.
Sponsor class is still included through sponsor_tier; lead_sponsor_canonical is target-encoded inside each training fold.


,target,model,train_rows,valid_rows,n_features,mae_original,rmse_original,mae_log,r2_log
0,country_count,enhanced_hgb_wide,15903,3976,31,2.5521,4.7429,0.4180,0.4942
1,country_count,enhanced_hgb,15903,3976,31,2.5702,4.7532,0.4208,0.4951
2,country_count,enhanced_xgb_depth4,15903,3976,31,2.5891,4.7777,0.4234,0.4951
3,country_count,enhanced_xgb_depth3,15903,3976,31,2.6300,4.8395,0.4301,0.4878
4,duration,enhanced_hgb_wide,16505,4127,31,9.8030,15.2147,0.3809,0.5972
5,duration,enhanced_hgb,16505,4127,31,9.9449,15.4170,0.3873,0.5890
6,duration,enhanced_xgb_depth4,16505,4127,31,9.9373,15.3123,0.3874,0.5903
7,duration,enhanced_xgb_depth3,16505,4127,31,10.0623,15.4876,0.3920,0.5820
8,enrollment,enhanced_hgb_wide,16346,4087,31,268.8015,1954.8195,0.5908,0.6383
9,enrollment,enhanced_hgb,16346,4087,31,270.9425,1962.2749,0.5938,0.6359


Best dependency-pruned model per target:


,target,model,mae_original,rmse_original,mae_log,r2_log,n_features
0,country_count,enhanced_hgb_wide,2.5521,4.7429,0.4180,0.4942,31
1,duration,enhanced_hgb_wide,9.8030,15.2147,0.3809,0.5972,31
2,enrollment,enhanced_hgb_wide,268.8015,1954.8195,0.5908,0.6383,31
3,site_count,enhanced_hgb,22.9127,56.0357,0.7489,0.5784,31


Training-derived ratio bounds used for reconciliation audit:


,ratio,p01,p99
0,patients_per_site,1.0000,763.6
1,sites_per_country,1.0000,75.0
2,patients_per_country,3.2333,1535.6


Independent prediction reconciliation flags:


,flag,n_flagged,pct_flagged
0,flag_below_min_enrollment,0,0.0000
1,flag_below_min_site_count,0,0.0000
2,flag_below_min_country_count,6,0.0137
3,flag_more_countries_than_sites,1,0.0023
4,flag_less_than_one_patient_per_site,0,0.0000
5,flag_patients_per_site_outside_train_range,1,0.0023
6,flag_sites_per_country_outside_train_range,1,0.0023
7,flag_patients_per_country_outside_train_range,1,0.0023


Worst reconciliation examples from dependency-pruned independent predictions:


,nct_id,pred_enrollment,actual_enrollment,pred_site_count,actual_site_count,pred_country_count,actual_country_count,pred_patients_per_site,pred_sites_per_country,pred_patients_per_country,flag_below_min_enrollment,flag_below_min_site_count,flag_below_min_country_count,flag_more_countries_than_sites,flag_less_than_one_patient_per_site,flag_patients_per_site_outside_train_range,flag_sites_per_country_outside_train_range,flag_patients_per_country_outside_train_range,n_reconciliation_flags
200,NCT04817111,16.320809,7.0,1.384898,1,1.518929,1.0,11.784842,0.911760,10.744943,False,False,False,True,False,False,True,False,2
18,NCT04583995,20239.069013,15185.0,23.011789,33,1.453852,1.0,879.508703,15.828153,13920.997888,False,False,False,False,False,True,False,True,2
365,NCT04661644,13.416657,20.0,1.235450,1,0.894990,1.0,10.859731,1.380406,14.990838,False,False,True,False,False,False,False,False,1
332,NCT01984528,30.553503,30.0,1.696434,2,0.980600,1.0,18.010423,1.729996,31.157961,False,False,True,False,False,False,False,False,1
212,NCT01654484,107.295791,60.0,3.741343,10,0.995692,1.0,28.678415,3.757531,107.760034,False,False,True,False,False,False,False,False,1
279,NCT05795517,157.259026,210.0,3.719693,1,0.996704,1.0,42.277420,3.731994,157.779087,False,False,True,False,False,False,False,False,1
34,NCT04435392,86.969330,107.0,1.673536,1,0.991871,1.0,51.967409,1.687251,87.682075,False,False,True,False,False,False,False,False,1
405,NCT06954766,185.175319,100.0,3.262853,1,0.969541,1.0,56.752572,3.365359,190.992769,False,False,True,False,False,False,False,False,1
412,NCT05626634,20.502361,41.0,14.214526,28,1.618609,2.0,1.442353,8.781938,12.666653,False,False,False,False,False,False,False,False,0
154,NCT01412424,44.333081,155.0,28.422307,36,6.984199,13.0,1.559799,4.069515,6.347625,False,False,False,False,False,False,False,False,0


#### <REF:STEP_3_VALIDATION>

> #### **18. Step 3 Validation Checklist**
> After running model benchmarks, validate the training dataset, feature setup, and first-pass results.
>
> Check:
> - Only low implausible duration/enrollment rows are excluded.
> - High global trials remain in the training data.
> - Target-specific observed fields are excluded from their own model features.
> - XGBoost availability is reported.
> - Feature counts look reasonable for each target.


#### <REF:TARGET_SAVE>

> #### **19. Optional Target Dataset Persistence**
> Save the derived model-target datasets under `data/processed/estimation/targets/`. This creates new estimation-specific inputs and does not modify the primary source data.
>
> Keep `SAVE_TARGET_DATASETS = False` until the target summaries are reviewed. Set it to `True` after validation.


In [14]:
# <REF:TARGET_SAVE_CODE>
SAVE_TARGET_DATASETS = False

if SAVE_TARGET_DATASETS:
    for name, data in model_target_datasets.items():
        out_path = TARGET_OUTPUT_DIR / f"estimation_{name}_model_targets.csv"
        data.to_csv(out_path, index=False)
        print(f"Saved {name}: {out_path}")
else:
    print("SAVE_TARGET_DATASETS is False. No target datasets written to disk yet.")
    print(f"When validated, outputs will be saved under: {TARGET_OUTPUT_DIR}")


SAVE_TARGET_DATASETS is False. No target datasets written to disk yet.
When validated, outputs will be saved under: /home/delaunan/code/delaunan/clintrialpredict/data/processed/estimation/targets


#### <REF:STEP_2_VALIDATION>

> #### **20. Step 2 Validation Checklist**
> Before model benchmarking, validate the target datasets.
>
> Check:
> - Duration target uses date-derived total trial duration, not `primary_duration_months`.
> - Duration target rows are close to the `is_completed_duration_target` count.
> - Enrollment target rows are close to the `is_completed_actual_enrollment_target` count.
> - Site and country target rows are close to their completed target flag counts.
> - Target medians and p95 values are plausible by clinical trial standards; total duration median should be materially longer than endpoint duration.
> - Target max values are reviewed in `<REF:TARGET_OUTLIER_AUDIT_CODE>` before training.
> - Proposed thresholds are validated before any exclusion or capping is applied.
> - `number_of_arms_num` correctly turns `UNKNOWN` into missing rather than invalid numeric values.
> - No derived target CSVs are written until `SAVE_TARGET_DATASETS` is intentionally set to `True`.


#### <REF:NEXT_STEP_TARGET_BUILD>

> #### **Next Step: Operational Target Construction**
> After Step 1 validation, the next block should create separate modelling datasets for:
>
> - final duration;
> - final enrollment;
> - final site count;
> - final country count.
>
> These datasets should be saved separately under `data/processed/estimation/` only after their target definitions are reviewed.